# WI vs IND — 2nd ODI 2026
## Match Analysis Notebook

**Match:** West Indies vs India, 2nd ODI  
**Date:** 2026-09-30  
**Venue:** Barsapara Cricket Stadium, Guwahati  
**Result:** India won by 8 wickets  
**Player of the Match:** Shubman Gill  

This notebook reads the raw ball-by-ball CSV (`IndiaVsWI_2ndODI.csv`), performs step-by-step analysis, and renders every chart inline.  
The final cells run the full ReportLab PDF generation script to produce `IndiaVsWI_2ndODI_analysis.pdf`.

---
### Notebook Structure
1. Imports & palette  
2. Load CSV & derive match metadata  
3. Pre-process deliveries (legal balls, wickets, boundaries)  
4. Stat helper functions  
5. Compute all aggregates  
6. Analytics helpers  
7–21. Individual charts (inline)  
22. PDF generation

## 1. Imports & Colour Palette

In [ ]:
# ── Magic command: render all matplotlib figures inline in the notebook ──────
%matplotlib inline

import io
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── File paths (forward-slashes work on Windows too) ─────────────────────────
BASE  = '.'
MATCH = 'IndiaVsWI_2ndODI'
CSV_PATH = f'{BASE}/{MATCH}.csv'
OUT_PDF  = f'{BASE}/{MATCH}_analysis.pdf'

# ── Matplotlib global style for dark-themed charts ───────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#FFFFFF',   # white background
    'axes.facecolor':    '#F5F7FA',   # very light panel
    'text.color':        '#222222',
    'axes.labelcolor':   '#222222',
    'xtick.color':       '#444444',
    'ytick.color':       '#444444',
    'axes.edgecolor':    '#CCCCCC',
    'grid.color':        '#CCCCCC',
    'grid.alpha':        0.5,
    'legend.facecolor':  '#FFFFFF',
    'legend.labelcolor': '#222222',
    'legend.edgecolor':  '#CCCCCC',
})

# ── Team colour palette (matplotlib hex strings) ─────────────────────────────
WI_MPL   = '#FF6348'   # West Indies maroon
IND_MPL  = '#2196F3'   # India blue
DARK_BG  = '#FFFFFF'   # page background
PANEL_BG = '#F5F7FA'   # panel background

print('Imports OK. Base path:', BASE)

## 2. Load CSV & Derive Match Metadata

The single-file CSV has one row per delivery.  
Key columns: `innings`, `ball`, `batting_team`, `striker`, `bowler`, `runs_off_bat`, `extras`, `wides`, `noballs`, `wicket_type`, `player_dismissed`.

In [ ]:
# ── Read the raw delivery-level CSV ──────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns')
print('Columns:', list(df_raw.columns))

# ── Derive metadata from the first row (no separate info CSV) ────────────────
# The CSV embeds match details like start_date and venue per delivery row.
_r0 = df_raw.iloc[0]
match_date    = str(_r0.get('start_date', '2026-09-30'))
venue         = str(_r0.get('venue', 'Barsapara Cricket Stadium, Guwahati'))

# Identify batting teams by splitting innings
_inn1_rows = df_raw[df_raw['innings'] == 1]
_inn2_rows = df_raw[df_raw['innings'] == 2]
team1 = str(_inn1_rows['batting_team'].iloc[0]) if not _inn1_rows.empty else 'West Indies'
team2 = str(_inn2_rows['batting_team'].iloc[0]) if not _inn2_rows.empty else 'India'

# Hard-coded match result facts (not in ball-by-ball CSV)
winner        = 'India'
winner_wkts   = 8          # India won by 8 wickets
winner_runs   = ''
toss_winner   = 'India'
toss_decision = 'bowl'
pom           = 'Shubman Gill'
match_no      = '2'

# Short abbreviations used on charts
_t1l = 'WI'   # team1 label
_t2l = 'IND'  # team2 label
_t1c = WI_MPL
_t2c = IND_MPL

print(f'Team 1 (bat first): {team1}')
print(f'Team 2 (bat second): {team2}')
print(f'Date: {match_date}  |  Venue: {venue}')
print(f'Result: {winner} won by {winner_wkts} wickets')

## 3. Pre-Processing

### What we compute
| Column | Meaning |
|---|---|
| `team_run` | Runs credited to the batting team (bat runs + extras) |
| `is_wide` | 1 if the delivery is a wide |
| `is_noball` | 1 if the delivery is a no-ball |
| `legal` | 1 if not wide AND not no-ball (counts as a faced delivery) |
| `is_wicket` | 1 if a wicket fell on this ball |
| `is_dot` | 1 if a legal delivery produces zero team runs |
| `is_4` / `is_6` | Boundary flags |
| `over_no` | 1-based over number (`ball='0.2'` → over 1) |
| `ball_no` | 1-based ball within the over |
| `phase` | ODI phase: Powerplay (1-10), Middle (11-40), Death (41-50) |

In [ ]:
# ── Work on a copy so df_raw stays untouched ─────────────────────────────────
df = df_raw.copy()
df.columns = df.columns.str.strip()  # remove any accidental leading/trailing spaces

# ── Ensure numeric columns are int (CSV may store them as float with NaN) ────
for c in ['runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)

# ── Derived delivery flags ────────────────────────────────────────────────────
df['team_run']  = df['runs_off_bat'] + df['extras']   # total runs to batting side
df['is_wide']   = (df['wides']   > 0).astype(int)
df['is_noball'] = (df['noballs'] > 0).astype(int)
df['legal']     = ((df['is_wide'] == 0) & (df['is_noball'] == 0)).astype(int)  # countable ball
df['is_wicket'] = df['wicket_type'].notna().astype(int)
df['is_dot']    = ((df['team_run'] == 0) & (df['legal'] == 1)).astype(int)
df['is_4']      = (df['runs_off_bat'] == 4).astype(int)
df['is_6']      = (df['runs_off_bat'] == 6).astype(int)

# ── Parse over notation to 1-based over_no and ball_no ───────────────────────
# actual_delivery format: '0.1' = over 1 ball 1, '43.3' = over 44 ball 3
def parse_over(o):
    parts = str(o).split('.')
    return int(parts[0]) + 1, int(parts[1])

df[['over_no', 'ball_no']] = df['actual_delivery'].apply(lambda x: pd.Series(parse_over(x)))

# ── ODI phase assignment ──────────────────────────────────────────────────────
# Powerplay: overs 1-10  |  Middle: 11-40  |  Death: 41-50
df['phase'] = df['over_no'].apply(
    lambda o: 'Powerplay' if o <= 10 else ('Middle' if o <= 40 else 'Death'))

# ── Split into innings ────────────────────────────────────────────────────────
inn1 = df[df['innings'] == 1].copy()   # team1 (WI) batting
inn2 = df[df['innings'] == 2].copy()   # team2 (IND) batting

print(f'Inn1 deliveries: {len(inn1)}  |  Inn2 deliveries: {len(inn2)}')
print(f'Inn1 wickets: {inn1["is_wicket"].sum()}  |  Inn2 wickets: {inn2["is_wicket"].sum()}')
df[['over_no', 'phase', 'team_run', 'legal', 'is_wicket', 'is_dot', 'is_4', 'is_6']].head()

## 4. Stat Helper Functions

Reusable functions that aggregate delivery-level data into scorecard rows.

In [ ]:
# ── Phase Stats  ──────────────────────────────────────────────────────────────
# Returns a list of dicts: one per phase with runs, wickets, RR, dot%
def phase_stats(inn):
    rows = []
    for ph in ['Powerplay', 'Middle', 'Death']:
        d = inn[inn['phase'] == ph]
        if d.empty:
            continue
        runs  = d['team_run'].sum()
        wkts  = int(d['is_wicket'].sum())
        legal = d['legal'].sum()
        overs = legal / 6
        rr    = runs / overs if overs > 0 else 0
        dotp  = d['is_dot'].sum() / legal * 100 if legal > 0 else 0
        rows.append({'Phase': ph, 'Runs': runs, 'Wkts': wkts,
                     'Overs': f'{int(legal//6)}.{int(legal%6)}',
                     'RR': f'{rr:.2f}', 'Dot%': f'{dotp:.0f}%'})
    return rows

# ── Batting Stats  ────────────────────────────────────────────────────────────
# Groups by striker; counts faced balls (excludes wides), SR, dot%
def bat_stats(inn):
    rows = []
    for bat, g in inn.groupby('striker'):
        faced = g[g['is_wide'] == 0].shape[0]  # balls faced = non-wide deliveries
        runs  = g['runs_off_bat'].sum()
        fours = int(g['is_4'].sum())
        sixes = int(g['is_6'].sum())
        dots  = g[(g['runs_off_bat'] == 0) & (g['is_wide'] == 0)].shape[0]
        sr    = runs / faced * 100 if faced > 0 else 0
        dotp  = dots / faced * 100 if faced > 0 else 0
        rows.append({'Batsman': bat, 'R': runs, 'B': faced, '4s': fours, '6s': sixes,
                     'SR': round(sr, 1), 'Dot%': round(dotp, 1)})
    rows.sort(key=lambda x: -x['R'])  # sort highest runs first
    return rows

# ── Bowling Stats  ────────────────────────────────────────────────────────────
# Groups by bowler; team_run includes all extras credited off that bowler
def bowl_stats(inn):
    rows = []
    for bwl, g in inn.groupby('bowler'):
        legal = g['legal'].sum()
        overs = legal / 6
        runs  = (g['runs_off_bat'] + g['wides'] + g['noballs']).sum()  # byes/leg-byes are not charged to bowler
        wkts  = int((g['is_wicket'] & ~g['wicket_type'].fillna('').str.lower().isin(['run out','retired hurt','retired out','obstructing the field'])).sum())
        econ  = runs / overs if overs > 0 else 0
        dotp  = g['is_dot'].sum() / legal * 100 if legal > 0 else 0
        wides = int(g['wides'].sum())
        nbs   = int(g['noballs'].sum())
        rows.append({'Bowler': bwl,
                     'O': f'{int(legal//6)}.{int(legal%6)}',
                     'R': runs, 'W': wkts, 'Econ': round(econ, 2),
                     'Dot%': round(dotp, 1), 'Wd': wides, 'NB': nbs})
    rows.sort(key=lambda x: -x['W'])  # most wickets first
    return rows

# ── Cumulative runs over time (for scoring-progression line chart) ────────────
# Returns list of (overs_elapsed, cumulative_runs) points
def cumulative_runs(inn):
    pts = [(0.0, 0)]
    cum = 0
    legal_count = 0
    for _, row in inn.iterrows():
        cum += int(row['team_run'])
        if row['legal']:
            legal_count += 1
            pts.append((round(legal_count / 6, 3), cum))
    return pts

# ── Per-over cumulative run rate ──────────────────────────────────────────────
# For each over, calculates RR using cumulative runs up to that point
def rr_per_over(inn):
    rr, cum_r, cum_l = {}, 0, 0
    for ov, g in inn.groupby('over_no'):
        cum_r += g['team_run'].sum()
        cum_l += g['legal'].sum()
        rr[ov] = cum_r / (cum_l / 6) if cum_l > 0 else 0
    return rr

# ── Name formatter  ───────────────────────────────────────────────────────────
# 'Shubman Gill' → 'S.Gill'; handles single-word names gracefully
def fmt_name(name):
    parts = str(name).strip().split()
    return (parts[0][0].upper() + '.' + parts[-1]) if len(parts) >= 2 else (parts[0] if parts else str(name))

# ── Fall of Wickets  ──────────────────────────────────────────────────────────
# Returns list of {Wkt, Score, Over, Batsman} in order of dismissal
def fall_of_wickets(inn):
    fow = []
    cum_r = 0; cum_l = 0; wkt_n = 0
    pd_col = 'player_dismissed' if 'player_dismissed' in inn.columns else 'striker'
    for _, row in inn.iterrows():
        cum_r += int(row['team_run'])
        if row['legal']:
            cum_l += 1
        if row['is_wicket']:
            wkt_n += 1
            ov_str = f'{cum_l//6}.{cum_l%6}'
            dismissed = str(row.get(pd_col, row['striker']))
            dismissed = '?' if dismissed == 'nan' else fmt_name(dismissed)
            fow.append({'Wkt': wkt_n, 'Score': cum_r, 'Over': ov_str, 'Batsman': dismissed})
    return fow

# ── Partnerships  ─────────────────────────────────────────────────────────────
# Segments the innings between wickets; identifies the pair at the crease
def partnerships(inn):
    balls = inn.reset_index(drop=True)
    has_ns = 'non_striker' in balls.columns
    parts = []
    start = 0
    wkt_no = 1
    wicket_indices = list(balls[balls['is_wicket'] == 1].index)
    endpoints = wicket_indices + ([len(balls)-1] if (not wicket_indices or wicket_indices[-1] != len(balls)-1) else [])
    for end_idx in endpoints:
        seg = balls.iloc[start:end_idx+1]  # include the wicket/scoring delivery
        if seg.empty:
            start = end_idx + 1
            continue
        runs = int(seg['team_run'].sum())
        b_cnt = int(seg['legal'].sum())
        names = []
        for col in ['striker', 'non_striker'] if has_ns else ['striker']:
            for name in seg[col]:
                if pd.notna(name) and str(name).strip() and name not in names:
                    names.append(name)
        bats = [fmt_name(x) for x in names[:2]]
        while len(bats) < 2:
            bats.append('?')
        parts.append({'Wkt': wkt_no, 'Bat1': bats[0], 'Bat2': bats[1],
                      'Runs': runs, 'Balls': b_cnt,
                      'RR': round(runs / (b_cnt / 6), 1) if b_cnt > 0 else 0.0})
        start = end_idx + 1
        wkt_no += 1
    return sorted(parts, key=lambda x: -x['Runs'])

print('All helper functions defined.')

## 5. Compute All Aggregates

In [ ]:
# ── Scorecards ────────────────────────────────────────────────────────────────
ps1  = phase_stats(inn1)     # WI phase scorecard
ps2  = phase_stats(inn2)     # IND phase scorecard
bs1  = bat_stats(inn1)       # WI batting scorecard
bs2  = bat_stats(inn2)       # IND batting scorecard
bw1  = bowl_stats(inn2)      # WI bowlers bowling against IND (inn2)
bw2  = bowl_stats(inn1)      # IND bowlers bowling against WI (inn1)

# ── Progression series (for line charts) ─────────────────────────────────────
cum1 = cumulative_runs(inn1)
cum2 = cumulative_runs(inn2)
rr1  = rr_per_over(inn1)
rr2  = rr_per_over(inn2)

# ── Fall of wickets & partnerships ───────────────────────────────────────────
fow1 = fall_of_wickets(inn1)
fow2 = fall_of_wickets(inn2)
par1 = partnerships(inn1)
par2 = partnerships(inn2)

# ── High-level match totals ───────────────────────────────────────────────────
total1  = inn1['team_run'].sum();    wkts1  = inn1['is_wicket'].sum()
total2  = inn2['team_run'].sum();    wkts2  = inn2['is_wicket'].sum()
legal1  = inn1['legal'].sum();       legal2 = inn2['legal'].sum()
ov1_str = f"{int(legal1//6)}.{int(legal1%6)}"
ov2_str = f"{int(legal2//6)}.{int(legal2%6)}"

print(f'{_t1l}: {total1}/{int(wkts1)} in {ov1_str} overs')
print(f'{_t2l}: {total2}/{int(wkts2)} in {ov2_str} overs')
print(f'Target: {int(total1)+1}  |  Result: {winner} won by {winner_wkts} wkts')

## 6. Analytics Helper Functions

Extra derived metrics: boundary %, death-over finishers, wicket clusters, powerplay stats, dot-ball % by phase.

In [ ]:
# ── Boundary % of total runs ──────────────────────────────────────────────────
def boundary_pct(inn):
    b_runs = (inn['is_4'] * 4 + inn['is_6'] * 6).sum()  # runs from 4s and 6s
    total  = inn['team_run'].sum()
    return round(b_runs / total * 100, 1) if total > 0 else 0.0

# ── Death-overs batsmen (overs 41-50, min 6 balls) ───────────────────────────
def death_bat_rows(inn):
    d = inn[(inn['over_no'] >= 41) & (inn['is_wide'] == 0)].copy()
    if d.empty:
        return pd.DataFrame(columns=['striker', 'runs', 'balls'])
    g = d.groupby('striker').agg(runs=('runs_off_bat', 'sum'), balls=('legal', 'sum')).reset_index()
    g = g[g['balls'] >= 6].copy()
    g['sr'] = (g['runs'] / g['balls'] * 100).round(1)
    return g.sort_values('sr', ascending=False)

# ── Wicket-taker summary ──────────────────────────────────────────────────────
def wkt_taker_rows(inn):
    w = inn[inn['is_wicket'] == 1].copy()
    if w.empty:
        return pd.DataFrame(columns=['bowler', 'wickets'])
    return w.groupby('bowler').size().reset_index(name='wickets').sort_values('wickets', ascending=False)

# ── Find the biggest wicket cluster (most wickets in consecutive overs) ───────
def find_cluster(inn):
    wbo = inn[inn['is_wicket'] == 1].groupby('over_no')['is_wicket'].sum()
    if wbo.empty:
        return 0, 0, 0
    ovs = sorted(wbo.index.tolist())
    best_wk, best_start, best_runs = 0, 0, 0
    i = 0
    while i < len(ovs):
        j = i
        while j < len(ovs) - 1 and ovs[j+1] - ovs[j] <= 1:  # consecutive overs
            j += 1
        span = ovs[i:j+1]
        twk = int(sum(wbo[o] for o in span))
        tru = int(inn[inn['over_no'].isin(span)]['team_run'].sum())
        if twk > best_wk:
            best_wk, best_start, best_runs = twk, span[0], tru
        i = j + 1
    return best_start, best_wk, best_runs

# ── Dot-ball percentage per phase ─────────────────────────────────────────────
def dot_pct_by_phase(inn):
    result = {}
    for ph in ['Powerplay', 'Middle', 'Death']:
        d   = inn[inn['phase'] == ph]
        leg = d['legal'].sum()
        result[ph] = round(d['is_dot'].sum() / leg * 100, 1) if leg > 0 else 0.0
    leg = inn['legal'].sum()
    result['Total'] = round(inn['is_dot'].sum() / leg * 100, 1) if leg > 0 else 0.0
    return result

# ── Compute all analytics values ──────────────────────────────────────────────
bp1   = boundary_pct(inn1);  bp2   = boundary_pct(inn2)
dbr1  = death_bat_rows(inn1); dbr2 = death_bat_rows(inn2)
wtr1  = wkt_taker_rows(inn2)  # WI bowlers who got wickets
wtr2  = wkt_taker_rows(inn1)  # IND bowlers who got wickets
cl1_ov, cl1_wk, cl1_ru = find_cluster(inn1)
cl2_ov, cl2_wk, cl2_ru = find_cluster(inn2)
dp1   = dot_pct_by_phase(inn1)
dp2   = dot_pct_by_phase(inn2)

# Powerplay summary (first 10 overs)
pp1_s = next((p for p in ps1 if p['Phase'] == 'Powerplay'), None)
pp2_s = next((p for p in ps2 if p['Phase'] == 'Powerplay'), None)
pp1_runs  = int(pp1_s['Runs']) if pp1_s else 0
pp1_wkts  = int(pp1_s['Wkts']) if pp1_s else 0
pp2_runs  = int(pp2_s['Runs']) if pp2_s else 0
pp2_wkts  = int(pp2_s['Wkts']) if pp2_s else 0

print(f'Boundary %:  WI={bp1}%  IND={bp2}%')
print(f'Powerplay:   WI={pp1_runs}/{pp1_wkts}  IND={pp2_runs}/{pp2_wkts}')
print(f'Dot%:        WI={dp1["Total"]}%  IND={dp2["Total"]}%')

## 7. Scoring Progression

Line chart of cumulative runs vs overs for both innings.  
`▼` markers show wicket moments. Dashed gold line = par (linear scoring to target).

In [ ]:
target = int(total1) + 1   # IND needed to score 1 more than WI

fig, ax = plt.subplots(figsize=(13, 5))

# Build (overs, runs) series for each innings
x1 = [p[0] for p in cum1]; y1 = [p[1] for p in cum1]
x2 = [p[0] for p in cum2]; y2 = [p[1] for p in cum2]

# Plot cumulative run lines
ax.plot(x1, y1, color=WI_MPL,  lw=2.5, label=f'{team1} ({total1}/{int(wkts1)})')
ax.plot(x2, y2, color=IND_MPL, lw=2.5, label=f'{team2} ({total2}/{int(wkts2)})')

# Par line: linear path to reach the target in 50 overs
ax.plot([0, 50], [0, target], color='#E67E22', lw=1.3, ls='--', alpha=0.65,
        label=f'Par line (target {target})')

# Plot wicket markers as downward triangles at the over/score they fell
for _, row in inn1[inn1['is_wicket'] == 1].iterrows():
    lb  = inn1.loc[:row.name, 'legal'].sum()   # legal balls up to this delivery
    run = inn1.loc[:row.name, 'team_run'].sum()
    ax.scatter(lb / 6, run, color=WI_MPL,  s=70, zorder=5, marker='v')

for _, row in inn2[inn2['is_wicket'] == 1].iterrows():
    lb  = inn2.loc[:row.name, 'legal'].sum()
    run = inn2.loc[:row.name, 'team_run'].sum()
    ax.scatter(lb / 6, run, color=IND_MPL, s=70, zorder=5, marker='v')

# Phase boundary markers
for xv, lbl in [(10.5, 'PP end'), (40.5, 'Middle end')]:
    ax.axvline(xv, color='#AAAAAA', lw=0.9, ls=':', alpha=0.5)
    ax.text(xv + 0.2, ax.get_ylim()[1] * 0.02, lbl, color='#666666', fontsize=8)

ax.set_xlim(0, 50); ax.set_ylim(0)
ax.set_xlabel('Overs'); ax.set_ylabel('Runs')
ax.set_title('Scoring Progression  (▼ = wicket)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(pad=0.5)
plt.show()

### Insights — Scoring Progression
- **India's chase was controlled throughout**: their cumulative runs tracked slightly behind WI for the first ~18 overs, then surged ahead and never looked back.
- **WI's 405/7 was the highest total of this series**, yet India overhauled it with 39 balls (6.3 overs) to spare.
- **Wicket markers (▼)** reveal WI's mid-innings wobble: losing 4 wickets between overs 40–50 stalled momentum after a dominant start.
- **India's wickets** (just 2 in 43.3 overs) show the chase was never under real pressure — a flat, ascending line with almost no disruption.
- The **par line** crossing India's actual score around over 40 illustrates that India beat the required rate comfortably from start to finish.


## 8. Cumulative Run Rate by Over

Shows how the overall run rate evolved over each innings. Useful for identifying acceleration/collapse phases.

In [ ]:
overs = list(range(1, 51))

# Build per-over cumulative RR lists; default 0 if the over was not played
rr1v = [rr1.get(o, 0) for o in overs]
rr2v = [rr2.get(o, 0) for o in overs]

fig, ax = plt.subplots(figsize=(13, 4.5))

ax.plot(overs, rr1v, color=WI_MPL,  lw=2.5, marker='o', ms=3, label=team1)
ax.plot(overs, rr2v, color=IND_MPL, lw=2.5, marker='s', ms=3, label=team2)
ax.axhline(6.0, color='#222222', lw=1, ls='--', alpha=0.35, label='6 RPO reference')

ax.set_xlim(1, 50); ax.set_ylim(0)
ax.set_xlabel('Over'); ax.set_ylabel('Cumulative Run Rate')
ax.set_title('Cumulative Run Rate by Over', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(pad=0.5)
plt.show()

### Insights — Cumulative Run Rate by Over
- **WI's powerplay was their peak**: cumulative RR of 9.60 after 10 overs, driven by Campbell (101 off 67) and Hope's attacking start.
- WI's RR **dipped progressively** in the middle and death overs (middle: 7.77 RPO, death: 7.60 RPO) — the team never fully capitalised on the PP platform.
- **India matched WI's powerplay tempo** (9.20 RPO, 0 wickets) and then **exceeded it** in the middle overs (9.33 RPO) — by far the more dominant phase performance.
- India's **end-of-innings cumulative RR (~9.34)** beat WI's final RR (~8.10), meaning India scored at a faster overall clip even while winning.
- The **two lines crossing** around over 25 is the visual turning point: India's trajectory becomes steeper and never drops below WI's.


## 9. Runs Per Over (Bar Chart)

Side-by-side bar charts for each innings. Red `W` annotations appear over overs where wickets fell.

In [ ]:
overs = list(range(1, 51))

# Sum runs and count wickets for each over in each innings
r1 = [inn1[inn1['over_no'] == o]['team_run'].sum() for o in overs]
r2 = [inn2[inn2['over_no'] == o]['team_run'].sum() for o in overs]
w1 = [inn1[inn1['over_no'] == o]['is_wicket'].sum() for o in overs]
w2 = [inn2[inn2['over_no'] == o]['is_wicket'].sum() for o in overs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, runs, wkts, color, label in [
        (ax1, r1, w1, WI_MPL,  team1),
        (ax2, r2, w2, IND_MPL, team2)]:

    bars = ax.bar(overs, runs, color=color, alpha=0.85, zorder=3)

    # Annotate overs with wickets using a red 'W'
    for bar, w in zip(bars, wkts):
        if w > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    'W' * int(w), ha='center', va='bottom',
                    color='#FF6B6B', fontsize=7, fontweight='bold')

    ax.set_xlabel('Over'); ax.set_ylabel('Runs')
    ax.set_title(f'{label} — Runs per Over', fontsize=10, fontweight='bold')
    ax.set_xticks(list(range(5, 51, 5)))
    ax.set_xlim(0.5, 50.5)
    ax.grid(axis='y', alpha=0.2, zorder=0)

plt.tight_layout(pad=0.5)
plt.show()

### Insights — Runs Per Over
- **WI's highest over was Over 7 (22 runs)**: a powerplay blitz by Campbell and Hope that set the early tone.
- WI showed **inconsistency in the death** (overs 41–50): multiple single-digit overs as 4 wickets fell, limiting the final total.
- **India's highest over was Over 29 (25 runs)**: Gill and Rohit at their fluent best in the middle phase.
- India had **fewer "collapse" overs** — no over with fewer than 3 runs, and their per-over distribution was flatter and more consistent.
- The red `W` markers on WI's chart in overs 40–50 tell the story: late wickets derailed a 420+ total bid.


## 10. Phase-wise Runs Comparison

Grouped bar chart: Powerplay / Middle / Death overs for each team.

In [ ]:
phases = ['Powerplay', 'Middle', 'Death']

# Look up runs per phase from pre-computed phase stats dicts
r1d = {p['Phase']: p['Runs'] for p in ps1}
r2d = {p['Phase']: p['Runs'] for p in ps2}
runs1 = [r1d.get(p, 0) for p in phases]
runs2 = [r2d.get(p, 0) for p in phases]

x = np.arange(len(phases)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 4.5))

b1 = ax.bar(x - w/2, runs1, w, label=team1, color=WI_MPL,  alpha=0.9, zorder=3)
b2 = ax.bar(x + w/2, runs2, w, label=team2, color=IND_MPL, alpha=0.9, zorder=3)

# Annotate each bar with its run total
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', va='bottom',
            color='#222222', fontsize=11, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(phases, fontsize=11)
ax.set_ylabel('Runs')
ax.set_title('Phase-wise Runs Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(pad=0.5)
plt.show()

### Insights — Phase-wise Runs
- **Middle overs (11–40) were the decisive battleground**: India scored 280/2 vs WI's 233/2 — 47 more runs for the *same* wicket count.
- **Powerplay was nearly even**: WI 96/1 vs India 92/0 — but India's *zero* powerplay wickets was the crucial difference (platform for Gill–Rohit stand).
- **WI's death overs unravelled**: 76/4 in 10 overs — they were set for 430+ before losing 4 late wickets.
- **India did not even need a full death innings** (batted only 3.3 overs, 34/0) — they had already won.
- The phase comparison starkly shows India dominated every single segment of the match, both with bat and implicitly with ball.


## 11. Dot-Ball % by Phase

Higher dot-ball % = more pressure exerted by the bowling side. Total column shows overall dot %.

In [ ]:
phases_ext = ['Powerplay', 'Middle', 'Death', 'Total']
x = np.arange(len(phases_ext)); w = 0.35

fig, ax = plt.subplots(figsize=(9, 4.5))

# dp1 / dp2 are dicts keyed by phase name (including 'Total')
b1 = ax.bar(x - w/2, [dp1[p] for p in phases_ext], w, label=_t1l, color=WI_MPL,  alpha=0.9, zorder=3)
b2 = ax.bar(x + w/2, [dp2[p] for p in phases_ext], w, label=_t2l, color=IND_MPL, alpha=0.9, zorder=3)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5, f'{h:.0f}%',
            ha='center', va='bottom', color='#222222', fontsize=9, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(phases_ext, fontsize=10)
ax.set_ylabel('Dot Ball %')
ax.set_title('Dot Ball % by Phase', fontsize=13, fontweight='bold')
ax.set_ylim(0, 80)
ax.legend(fontsize=9)
plt.tight_layout(pad=0.5)
plt.show()

### Insights — Dot-Ball % by Phase
- **India's powerplay dot% was 50.0%** — WI bowlers tied down Indian openers, yet scored 0 wickets; pressure without reward.
- **India's middle-over dot% (28.9%)** was the lowest of any phase/team combination — Gill and Rohit were in almost perfect command.
- **India's death dot% was just 19.0%** (best of any segment): in the 3.3 overs they batted, virtually every ball was scored off.
- **WI's overall dot% (37.7%)** is higher than India's (33.0%), meaning India converted more scoring opportunities per legal ball faced.
- WI's death dot% (43.3%) for their own batsmen shows India's bowlers successfully reduced their boundary options late in the innings.


## 12. Top Batsmen — Runs vs Balls

Horizontal bar chart showing runs (team colour) and balls faced (grey) for the top 6 batsmen per innings.

In [ ]:
top1 = bs1[:6]; top2 = bs2[:6]   # take top 6 by runs for each team

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, rows, color, label in [
        (ax1, top1, WI_MPL,  team1),
        (ax2, top2, IND_MPL, team2)]:

    names  = [fmt_name(r['Batsman']) for r in rows]
    runs_v = [r['R'] for r in rows]
    balls  = [r['B'] for r in rows]
    y = np.arange(len(names))

    # Paired horizontal bars: runs on top row, balls on bottom row
    ax.barh(y + 0.2, runs_v, 0.35, color=color, alpha=0.9, label='Runs')
    ax.barh(y - 0.2, balls,  0.35, color='#BBBBBB',  alpha=0.7, label='Balls')

    # Annotate with values
    for i, (r, b) in enumerate(zip(runs_v, balls)):
        ax.text(r + 0.5, i + 0.2, str(r), va='center', color='#222222', fontsize=9, fontweight='bold')
        ax.text(b + 0.5, i - 0.2, str(b), va='center', color='#E0E0E0',   fontsize=8)

    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Count')
    ax.set_title(f'{label} — Top Batsmen', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout(pad=0.5)
plt.show()

### Insights — Top Batsmen
- **WI produced three centurions**: Jangoo 114(77) SR 148, Hope 104(94) SR 111, Campbell 101(68) SR 149 — a remarkable collective effort.
- **India's answer was Shubman Gill: 223*(133) SR 168** — one of the great chasing innings in ODI history, with 26 fours and 8 sixes.
- **Rohit Sharma's 101(75) SR 135** complemented Gill perfectly; their 255-run opening stand made the result inevitable.
- Despite WI's 3 centuries, the **strike rates tell the story**: Gill's 169 beat every WI batsman's SR and his 223 nearly doubled Hope's 104.
- India's **lower order never needed to bat** — the chart's shorter bars for Gaikwad (31) and Kohli (29) are effectively cameos in a won game.


## 13. Fall of Wickets & Partnerships

Four-panel chart:  
- Top-left/right: Fall of wickets per team  
- Bottom-left/right: Top partnerships (by runs) per team

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 2, hspace=0.5, wspace=0.35)
ax1 = fig.add_subplot(gs[0, 0])   # WI FOW
ax2 = fig.add_subplot(gs[0, 1])   # IND FOW
ax3 = fig.add_subplot(gs[1, 0])   # WI partnerships
ax4 = fig.add_subplot(gs[1, 1])   # IND partnerships

for ax in [ax1, ax2, ax3, ax4]:
    ax.set_facecolor(PANEL_BG)
    for sp in ax.spines.values():
        sp.set_edgecolor('#EBEBF0')

# ── Fall of Wickets panel helper ──────────────────────────────────────────────
def plot_fow(ax, fow, color, title):
    if not fow:
        ax.text(0.5, 0.5, 'All Out / No Data', ha='center', va='center',
                color='#222222', transform=ax.transAxes, fontsize=10)
        ax.set_title(title, fontsize=10, fontweight='bold'); return
    scores = [f['Score'] for f in fow]
    labels = [f"W{f['Wkt']}: {f['Batsman']}" for f in fow]
    overs  = [f['Over']  for f in fow]
    y = range(len(fow))
    ax.barh(list(y), scores, color=color, alpha=0.8, height=0.6)
    max_s = max(scores) if scores else 1
    for i, (s, ov) in enumerate(zip(scores, overs)):
        ax.text(s + max_s * 0.02, i, f'{s} (ov {ov})', va='center', color='#222222', fontsize=7)
    ax.set_yticks(list(y)); ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Runs at Dismissal')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=4)

# ── Partnerships panel helper ─────────────────────────────────────────────────
def plot_parts(ax, parts, color, title):
    if not parts:
        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                color='#222222', transform=ax.transAxes, fontsize=10)
        ax.set_title(title, fontsize=10, fontweight='bold'); return
    top    = parts[:6]   # show top 6 partnerships by runs
    labels = [f"P{p['Wkt']}: {p['Bat1']} & {p['Bat2']}" for p in top]
    runs   = [p['Runs'] for p in top]
    y = range(len(top))
    ax.barh(list(y), runs, color=color, alpha=0.8, height=0.6)
    max_r = max(runs) if runs else 1
    for i, (r, p) in enumerate(zip(runs, top)):
        ax.text(r + max_r * 0.02, i, f"{r} ({p['Balls']}b, RR {p['RR']})",
                va='center', color='#222222', fontsize=7)
    ax.set_yticks(list(y)); ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Partnership Runs')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=4)

plot_fow(ax1,  fow1, WI_MPL,  f'{_t1l} — Fall of Wickets')
plot_fow(ax2,  fow2, IND_MPL, f'{_t2l} — Fall of Wickets')
plot_parts(ax3, par1, WI_MPL, f'{_t1l} — Top Partnerships')
plot_parts(ax4, par2, IND_MPL, f'{_t2l} — Top Partnerships')

plt.show()

### Insights — Fall of Wickets & Partnerships
- **WI's 3rd-wicket stand of 182 (Jangoo–Hope)** was the backbone of their innings — it consumed 130 balls and spanned the powerplay through mid-innings.
- **India's opening partnership of 255 (Rohit–Gill)** is one of the largest chasing opening stands in ODI history and effectively decided the match.
- WI's wickets clustering in the **death overs (4 in overs 40–50)** show the innings lost shape precisely when it needed a late flourish.
- India needed only **3 partnerships** to complete the chase — Rohit–Gill 255, then Kohli–Gill 65, then Gaikwad–Gill 86. Gill featured in all three.
- The **partnership chart** visually shows India's dominance: one massive bar dwarfs everything else on the page.


## 14. Bowling Scorecards — Wickets & Economy

In [ ]:
top_bw1 = sorted(bw1, key=lambda x: -x['W'])[:6]   # WI top bowlers vs IND
top_bw2 = sorted(bw2, key=lambda x: -x['W'])[:6]   # IND top bowlers vs WI

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, rows, color, label in [
        (ax1, top_bw1, WI_MPL,  f'{team1} Bowlers (vs {team2})'),
        (ax2, top_bw2, IND_MPL, f'{team2} Bowlers (vs {team1})'  )]:

    names = [fmt_name(r['Bowler']) for r in rows]
    wkts  = [r['W']    for r in rows]
    econ  = [r['Econ'] for r in rows]
    y = np.arange(len(names))

    ax.bar(y - 0.2, wkts, 0.35, color=color, alpha=0.9, label='Wickets')
    ax.bar(y + 0.2, econ, 0.35, color='#BBBBBB',  alpha=0.7, label='Economy')

    for i, (ww, e) in enumerate(zip(wkts, econ)):
        ax.text(i - 0.2, ww + 0.05, str(ww), ha='center', va='bottom',
                color='#222222', fontsize=9, fontweight='bold')
        ax.text(i + 0.2, e  + 0.05, f'{e:.1f}', ha='center', va='bottom',
                color='#E0E0E0', fontsize=8)

    ax.set_xticks(y); ax.set_xticklabels(names, fontsize=9)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout(pad=0.5)
plt.show()

### Insights — Bowling Scorecards
- **India's best: Kuldeep Yadav 2/65 Economy 6.50** — the only bowler across both teams with both multiple wickets *and* reasonable economy.
- **Gurnoor Brar took 2 wickets but cost 97 runs** (Economy 9.70) — effective at removing batsmen but expensive.
- **WI's bowling was comprehensively outclassed**: best economy was Roston Chase (7.41) with *zero* wickets; Alzarri Joseph conceded 0/89 at 11.12.
- **Keemo Paul's 3 overs cost 49 runs (Economy 16.33)** — a brutal return that exemplifies WI's inability to contain Gill and Rohit.
- Jadeja (2/75, Econ 7.50) and Krishna (1/45, Econ 9.00) show that even India's effective bowlers leaked runs on a flat pitch.


## 15. Economy vs Wickets Scatter

Each dot is a bowler. Ideal zone = right (high wickets) and below 6 RPO dashed line.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, bw, color, label in [
        (ax1, bw1, WI_MPL,  f'{team1} Bowlers'),
        (ax2, bw2, IND_MPL, f'{team2} Bowlers')]:

    wkts  = [b['W']    for b in bw]
    econ  = [b['Econ'] for b in bw]
    names = [fmt_name(b['Bowler']) for b in bw]

    # Scatter: x = wickets taken, y = economy rate
    ax.scatter(wkts, econ, c=color, s=130, alpha=0.85, zorder=3,
               edgecolors='white', linewidths=0.5)

    # Label each dot with the bowler's short name
    for w, e, n in zip(wkts, econ, names):
        ax.annotate(n, (w, e), textcoords='offset points', xytext=(5, 4),
                    color='#222222', fontsize=8)

    ax.axhline(6.0, color='#BBBBBB', lw=1, ls='--', alpha=0.5, label='6 RPO')
    ax.set_xlabel('Wickets'); ax.set_ylabel('Economy Rate')
    ax.set_title(f'{label}\nEconomy vs Wickets', fontsize=10, fontweight='bold')
    if wkts:
        ax.set_xticks(range(0, max(wkts) + 2))
    ax.legend(fontsize=8)

plt.tight_layout(pad=0.5)
plt.show()

### Insights — Economy vs Wickets Scatter
- The **ideal zone** (top-right: high wickets, low economy) is empty for *both* attacks — this was not a match for bowlers.
- **Kuldeep Yadav** is the sole bowler in a relatively productive quadrant: 2 wickets at 6.50 economy — far right of the WI bowling cluster.
- All **WI bowlers** sit above 7.0 economy and mostly at 0–1 wickets — the scatter shows a team that could neither contain nor dismiss India.
- The **horizontal spread** of WI bowlers (all at similar low wicket count) vs India's slight vertical spread (Kuldeep, Brar, Jadeja at 2 each) shows India's bowling was at least wicket-taking, even if expensive.
- **Keemo Paul's dot** (0 wickets, 16.33 economy) is an extreme outlier — the most punished spell of the match.


## 16. Match Analytics — 4-Panel Overview

1. Powerplay battle (runs/wickets in first 10 overs)  
2. Boundary contribution % (runs from 4s & 6s vs total)  
3. Death-over finishers (strike rate in overs 41-50, min 6 balls)  
4. Wicket distribution across bowlers

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = fig.add_gridspec(2, 2, hspace=0.55, wspace=0.38)
ax_pp = fig.add_subplot(gs[0, 0])   # Powerplay
ax_bp = fig.add_subplot(gs[0, 1])   # Boundary %
ax_df = fig.add_subplot(gs[1, 0])   # Death finishers
ax_wk = fig.add_subplot(gs[1, 1])   # Wicket distribution

for ax in [ax_pp, ax_bp, ax_df, ax_wk]:
    ax.set_facecolor(PANEL_BG)
    for sp in ax.spines.values():
        sp.set_edgecolor('#EBEBF0')

# Panel 1: Powerplay runs/wickets
bars_pp = ax_pp.bar([0, 1], [pp1_runs, pp2_runs], color=[WI_MPL, IND_MPL], width=0.5)
for bar, wk in zip(bars_pp, [pp1_wkts, pp2_wkts]):
    ax_pp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
               f'{int(bar.get_height())}/{wk}',
               ha='center', va='bottom', color='#222222', fontsize=11, fontweight='bold')
ax_pp.set_xticks([0, 1]); ax_pp.set_xticklabels([_t1l, _t2l], fontsize=12)
ax_pp.set_title('Powerplay Battle (Ov 1-10)\nRuns / Wickets', fontsize=11, pad=8)
ax_pp.set_ylabel('Runs')
ax_pp.set_ylim(0, max(pp1_runs, pp2_runs) * 1.3 + 5)

# Panel 2: Boundary % of total runs
bars_bp = ax_bp.bar([0, 1], [bp1, bp2], color=[WI_MPL, IND_MPL], width=0.5)
for bar, val in zip(bars_bp, [bp1, bp2]):
    ax_bp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
               f'{val}%', ha='center', va='bottom', color='#222222', fontsize=11, fontweight='bold')
ax_bp.set_xticks([0, 1]); ax_bp.set_xticklabels([_t1l, _t2l], fontsize=12)
ax_bp.set_title('Boundary Contribution\n% of Total Runs from 4s & 6s', fontsize=11, pad=8)
ax_bp.set_ylabel('%')
bp_min = max(0, min(bp1, bp2) * 0.75)
bp_max = min(100, max(bp1, bp2) * 1.3)
ax_bp.set_ylim(bp_min, bp_max)

# Panel 3: Death-over finishers
all_df_rows = ([(r['striker'], r['sr'], r['runs'], r['balls'], WI_MPL)  for _, r in dbr1.iterrows()] +
               [(r['striker'], r['sr'], r['runs'], r['balls'], IND_MPL) for _, r in dbr2.iterrows()])
all_df_rows.sort(key=lambda x: x[1], reverse=True); all_df_rows = all_df_rows[:8]
if all_df_rows:
    names_df = [fmt_name(x[0]) for x in all_df_rows]
    y_df     = range(len(all_df_rows))
    bars_df  = ax_df.barh(list(y_df), [x[1] for x in all_df_rows],
                          color=[x[4] for x in all_df_rows], height=0.6)
    max_sr = max(x[1] for x in all_df_rows)
    ax_df.set_xlim(0, max_sr * 1.28)
    for bar, rd in zip(bars_df, all_df_rows):
        ax_df.text(bar.get_width() + max_sr * 0.02, bar.get_y() + bar.get_height()/2,
                   f'{int(rd[2])}({int(rd[3])})', va='center', ha='left', color='#222222', fontsize=8)
    ax_df.set_yticks(list(y_df)); ax_df.set_yticklabels(names_df, fontsize=10)
    ax_df.invert_yaxis()
ax_df.set_title('Death Overs Finishers (Ov 41-50)\nStrike Rate  (min 6 balls)', fontsize=11, pad=8)
ax_df.set_xlabel('Strike Rate')
ax_df.axvline(100, color='#777777', linestyle='--', linewidth=0.8)

# Panel 4: Wicket distribution
all_wk_rows = ([(r['bowler'], r['wickets'], WI_MPL,  _t1l) for _, r in wtr1.iterrows()] +
               [(r['bowler'], r['wickets'], IND_MPL, _t2l) for _, r in wtr2.iterrows()])
all_wk_rows.sort(key=lambda x: x[1], reverse=True); all_wk_rows = all_wk_rows[:10]
if all_wk_rows:
    names_wk = [fmt_name(x[0]) for x in all_wk_rows]
    y_wk     = range(len(all_wk_rows))
    bars_wk  = ax_wk.barh(list(y_wk), [x[1] for x in all_wk_rows],
                          color=[x[2] for x in all_wk_rows], height=0.6)
    for bar, rd in zip(bars_wk, all_wk_rows):
        ax_wk.text(bar.get_width() + 0.06, bar.get_y() + bar.get_height()/2,
                   f'({rd[3]})', va='center', ha='left', color='#222222', fontsize=9, fontweight='bold')
    ax_wk.set_yticks(list(y_wk)); ax_wk.set_yticklabels(names_wk, fontsize=10)
    ax_wk.invert_yaxis()
    ax_wk.set_xticks(range(0, int(max(x[1] for x in all_wk_rows)) + 2))
ax_wk.set_title('Wicket Distribution\nBowlers with Scalps', fontsize=11, pad=8)
ax_wk.set_xlabel('Wickets')

plt.show()

### Insights — Match Analytics (4-Panel)
- **Powerplay battle** was nearly even in runs (WI 96/1 vs IND 92/0) — but India's *zero powerplay wickets* created the match-winning platform.
- **Boundary % was high for both**: WI 55.3% and India 62.6% — confirming a flat, high-scoring venue where bowlers had little assistance.
- **India's boundary % (62.6%)** means nearly 2 in every 3 runs came in fours and sixes — an extraordinarily attacking, boundary-centric chase.
- **Wicket distribution** shows WI's 7 wickets were spread across all phases with 4 clustering in the death — the team lost its shape at the worst moment.
- The **death-over SR panel** starkly shows WI's lower-order batsmen were dismissed early, dragging the SR down, while India batted freely in the 3.3 overs they needed.


## 17. Rolling Run Rate — Momentum Tracker

5-over rolling average of RR. Phase markers at over 10.5 (end of PP) and 40.5 (start of death).

In [ ]:
window = 5   # number of overs in the rolling window
overs  = list(range(1, 51))

# Compute per-over RR (not cumulative)
def over_rr_dict(inn):
    return {o: (inn[inn['over_no'] == o]['team_run'].sum() /
                max(inn[inn['over_no'] == o]['legal'].sum() / 6, 1e-9))
            for o in overs}

rr1_ov = over_rr_dict(inn1)
rr2_ov = over_rr_dict(inn2)

# Rolling average: for each over index i, average the window ending at i
def rolling_avg(d, w):
    vals = [d.get(o, 0) for o in overs]
    return [sum(vals[max(0, i - w + 1):i + 1]) / len(vals[max(0, i - w + 1):i + 1])
            for i in range(len(vals))]

r1 = rolling_avg(rr1_ov, window)
r2 = rolling_avg(rr2_ov, window)
y_top = max(max(r1) if r1 else 0, max(r2) if r2 else 0) * 1.15 + 1

fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(overs, r1, color=WI_MPL,  lw=2.5, marker='o', ms=3,
        label=f'{_t1l} ({window}-ov rolling RR)')
ax.plot(overs, r2, color=IND_MPL, lw=2.5, marker='s', ms=3,
        label=f'{_t2l} ({window}-ov rolling RR)')

# Shaded area under each line to make phases visually clear
ax.fill_between(overs, r1, alpha=0.12, color=WI_MPL)
ax.fill_between(overs, r2, alpha=0.12, color=IND_MPL)

# Phase boundary dotted lines at ODI breakpoints
for x_ph, lbl_ph in [(10.5, 'PP'), (40.5, 'Mid')]:
    ax.axvline(x_ph, color='#BBBBBB', lw=1, ls=':', alpha=0.6)
    ax.text(x_ph + 0.15, y_top * 0.88, lbl_ph, color='#666666', fontsize=8)

ax.axhline(6.0, color='#E67E22', lw=1, ls='--', alpha=0.5, label='6 RPO')
ax.set_xlim(1, 50); ax.set_ylim(0, y_top)
ax.set_xlabel('Over'); ax.set_ylabel('Rolling RR')
ax.set_title(f'{window}-Over Rolling Run Rate  —  Momentum Tracker',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(pad=0.5)
plt.show()

### Insights — 5-Over Rolling Run Rate
- **WI's rolling RR peaked** in the powerplay window (overs 6–10) at ~9.6+ then gradually declined through the middle — the team never replicated its opening burst.
- **India's rolling RR was remarkably consistent**: a relatively flat, elevated line through overs 5–43, rarely dipping below 8.5 RPO on a 5-over basis.
- The **phase dividers** at overs 10 and 40 show that India's middle-phase momentum (rolling RR hovering around 9.3) was higher than WI's (~8.5 in the same range).
- **WI's death dip** (rolling RR drops steeply from over 40) is clearly visible — 4 wickets in 10 overs disrupted rhythm when acceleration was most needed.
- India's line **never shows the kind of trough** WI experienced — a testament to the consistency of the Rohit–Gill partnership.


## 18. Chase Analysis — Actual vs Required Run Rate

Green fill = India ahead of requirement; Red fill = India behind.  
Only plotted up to the over the chase was completed.

In [ ]:
target = int(total1) + 1   # IND needed to beat WI's total

# Build a map of {over: cumulative_score} for inn2
cum = 0
cum_scores = {0: 0}
for o in sorted(inn2['over_no'].unique()):
    cum += inn2[inn2['over_no'] == o]['team_run'].sum()
    cum_scores[o] = int(cum)

max_ov = max(k for k in cum_scores if k > 0)
overs_plot = list(range(1, max_ov + 1))

act_rr, req_rr = [], []
for o in overs_plot:
    # Actual RR = cumulative score up to over o divided by overs bowled
    sc_keys  = [k for k in cum_scores if k <= o]
    sc       = cum_scores[max(sc_keys)] if sc_keys else 0
    # Required RR = runs still needed divided by overs remaining
    prev_keys = [k for k in cum_scores if k < o]
    prev     = cum_scores[max(prev_keys)] if prev_keys else 0
    act_rr.append(sc / o)
    req_rr.append((target - prev) / max(51 - o, 1))   # ODI: 50-over format

fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(overs_plot, act_rr, color=IND_MPL, lw=2.5, marker='o', ms=4,
        label=f'{_t2l} Actual RR')
ax.plot(overs_plot, req_rr, color='#E67E22', lw=2, ls='--', marker='s', ms=3,
        label='Required RR')

# Fill between lines: green when ahead, red when behind
for i in range(len(overs_plot) - 1):
    c = '#27AE60' if act_rr[i] >= req_rr[i] else '#E74C3C'
    ax.fill_between(overs_plot[i:i+2], act_rr[i:i+2], req_rr[i:i+2],
                    alpha=0.18, color=c)

ax.set_xlim(1, max_ov); ax.set_ylim(0)
ax.set_xlabel('Over'); ax.set_ylabel('Run Rate')
ax.set_title(f'{_t2l} Chase Analysis  —  Actual vs Required RR  (target {target})',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(pad=0.5)
plt.show()

### Insights — Chase Analysis (Actual vs Required RR)
- **India were ahead of required rate from approximately over 5 onwards** — the green (ahead) zone dominates the entire chart.
- The **required rate started at 8.12 RPO** (406 in 50 overs); India's actual rate consistently exceeded this thanks to the explosive opening stand.
- By **over 30**, India's required rate had fallen to ~6 RPO while their actual rate remained ~9+ — the chase was effectively over.
- **India crossed WI's total in over 44** — winning with 6.3 overs to spare, requiring no final-over heroics.
- The **tiny red zone** (India briefly behind) near overs 1–4 simply reflects the first few cautious deliveries — never a real danger sign.


## 19. Batsman Contribution (% of Team Total)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, bs, inn, color, label in [
        (ax1, bs1, inn1, WI_MPL,  _t1l),
        (ax2, bs2, inn2, IND_MPL, _t2l)]:

    ax.set_facecolor(PANEL_BG)
    t_total = inn['team_run'].sum()            # team total including extras
    extras  = int(t_total - inn['runs_off_bat'].sum())  # extras = team_run - bat_run
    top     = bs[:7]                           # top 7 batsmen by runs

    names   = [fmt_name(b['Batsman']) for b in top]
    runs_v  = [b['R'] for b in top]

    # Remaining bat runs not accounted for by the top 7
    other   = max(0, int(inn['runs_off_bat'].sum()) - sum(runs_v))
    all_runs  = runs_v + [other, extras]
    all_names = names  + ['Others', 'Extras']

    # Top 7 in team colour, Others in grey, Extras in darker grey
    bar_colors = [color] * len(top) + ['#AAAAAA', '#E0E0E8']

    pcts = [r / t_total * 100 if t_total > 0 else 0 for r in all_runs]
    y    = range(len(all_names))
    bars = ax.barh(list(y), pcts, color=bar_colors, height=0.7, alpha=0.9)

    for bar, run, pct in zip(bars, all_runs, pcts):
        if pct > 1.5:
            ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                    f'{int(run)} ({pct:.0f}%)', va='center', ha='left',
                    color='#222222', fontsize=8)

    ax.set_yticks(list(y)); ax.set_yticklabels(all_names, fontsize=9)
    ax.set_xlabel('% of Team Total')
    ax.set_title(f'{label} — Batsman Contribution', fontsize=10, fontweight='bold')
    ax.invert_yaxis()

plt.tight_layout(pad=0.5)
plt.show()

### Insights — Batsman Contribution (% of Team Total)
- **WI's top 3 batsmen** (Campbell 101, Hope 104, Jangoo 114) contributed **79% of the team's 405 runs** — extraordinary collective but dangerously top-heavy.
- **India's top 2** (Rohit 101, Gill 223) contributed **~80% of 406** — similar concentration but with even higher individual quality (Gill's 55% slice is historic for a chase).
- Both teams showed **minimal lower-order contribution** — WI's middle/lower order (Roston Chase, Shamar Joseph etc.) added only ~84 runs in 7 wickets, failing to build on the platform.
- India's **Gaikwad (7.6%) and Kohli (7.1%)** are supporting-cast contributions in a match decided at the top.
- The chart illustrates a recurring theme: **in modern ODIs, the top-order dominance is near-total** — whoever wins the opening-to-3rd-wicket battle, wins the match.


## 20. Boundary Distribution by Phase

Grouped bars: solid = 4s, hatched = 6s. Four bars per phase (WI 4s, WI 6s, IND 4s, IND 6s).

In [ ]:
phases = ['Powerplay', 'Middle', 'Death']

# Count 4s and 6s per phase per innings
def bnd_by_phase(inn):
    return ({p: int(inn[inn['phase'] == p]['is_4'].sum()) for p in phases},
            {p: int(inn[inn['phase'] == p]['is_6'].sum()) for p in phases})

f1, s1 = bnd_by_phase(inn1)
f2, s2 = bnd_by_phase(inn2)

x = np.arange(len(phases)); w = 0.2
fig, ax = plt.subplots(figsize=(11, 4.5))

# Four bar groups side by side per phase
b1 = ax.bar(x - 1.5*w, [f1[p] for p in phases], w, label=f'{_t1l} 4s', color=WI_MPL,  alpha=0.9)
b2 = ax.bar(x - 0.5*w, [s1[p] for p in phases], w, label=f'{_t1l} 6s', color=WI_MPL,  alpha=0.6, hatch='//')
b3 = ax.bar(x + 0.5*w, [f2[p] for p in phases], w, label=f'{_t2l} 4s', color=IND_MPL, alpha=0.9)
b4 = ax.bar(x + 1.5*w, [s2[p] for p in phases], w, label=f'{_t2l} 6s', color=IND_MPL, alpha=0.6, hatch='//')

# Annotate bars with count values
for bar in list(b1) + list(b2) + list(b3) + list(b4):
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, str(int(h)),
                ha='center', va='bottom', color='#222222', fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(phases, fontsize=11)
ax.set_ylabel('Count')
ax.set_title('Boundary Distribution by Phase  (4s solid, 6s hatched)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, ncol=2)
plt.tight_layout(pad=0.5)
plt.show()

### Insights — Boundary Distribution by Phase
- **Both teams scored the bulk of their boundaries in the Middle phase** (overs 11–40): WI 22 fours + 6 sixes; India 24 fours + 9 sixes — reflecting the flat, slow pitch.
- **India's powerplay boundaries** (10 fours + 5 sixes in 10 overs) were nearly as explosive as WI's PP (9 fours + 5 sixes) while losing no wickets.
- **India hit more sixes in every phase** (15 total vs WI's 12) — a sign of more aggressive, lofted shot-making particularly from Gill and Rohit.
- **WI's death boundary count collapsed** to just 7 fours + 1 six in 10 overs — confirming the wickets-driven stagnation in overs 41–50.
- India's **death boundary count** is inflated relative to deliveries faced (3.3 overs only) — every ball in those overs was played with freedom.


## 21. Dismissal Types

How were batsmen dismissed in each innings? (caught, bowled, lbw, run out, etc.)

In [ ]:
# Count dismissal types from wicket_type column
wk1 = inn1[inn1['is_wicket'] == 1]['wicket_type'].dropna().value_counts()
wk2 = inn2[inn2['is_wicket'] == 1]['wicket_type'].dropna().value_counts()

if wk1.empty and wk2.empty:
    print('No wicket data available.')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    for ax, wk, color, label in [(ax1, wk1, WI_MPL, _t1l), (ax2, wk2, IND_MPL, _t2l)]:
        ax.set_facecolor(PANEL_BG)
        if wk.empty:
            ax.text(0.5, 0.5, 'No wickets', ha='center', va='center',
                    color='#222222', transform=ax.transAxes, fontsize=10)
            ax.set_title(f'{label} — Dismissal Types', fontsize=10, fontweight='bold')
            continue
        y    = range(len(wk))
        bars = ax.barh(list(y), wk.values, color=color, alpha=0.85, height=0.6)
        for bar, val in zip(bars, wk.values):
            ax.text(bar.get_width() + 0.06, bar.get_y() + bar.get_height()/2,
                    str(val), va='center', ha='left', color='#222222', fontsize=9)
        ax.set_yticks(list(y)); ax.set_yticklabels(wk.index.tolist(), fontsize=9)
        ax.set_xlabel('Count')
        ax.set_title(f'{label} — Dismissal Types', fontsize=10, fontweight='bold')
        ax.invert_yaxis()

    plt.tight_layout(pad=0.5)
    plt.show()

### Insights — Dismissal Types
- **India lost only 2 wickets** in 43.3 overs — the dismissal-type chart is almost empty, reinforcing how controlled the chase was.
- **WI's 7 dismissals** were spread across multiple types, suggesting India's bowling attacked from various angles (seam movement, spin, pace variation).
- The **low wicket count for India** means the bowlers never found a consistent mode of dismissal — no bowler's signature delivery dominated.
- WI's wicket spread also shows **no single batsman "anchored" their innings beyond the top 3** — once the three centurions departed, the lower order had no answer.
- The dismissal chart is ultimately a **visual summary of the match**: one team lost 7 wickets under pressure; the other barely had to bat at all.


---
## Quick Text Summary

In [ ]:
# ── Print a concise text summary of key match stats ──────────────────────────
print('=' * 60)
print(f'  WI vs IND  |  2nd ODI  |  {match_date}')
print(f'  {venue}')
print('=' * 60)
print(f'  {team1}:  {total1}/{int(wkts1)} in {ov1_str} overs')
print(f'  {team2}:  {total2}/{int(wkts2)} in {ov2_str} overs')
print(f'  Result:   {winner} won by {winner_wkts} wickets')
print(f'  POM:      {pom}')
print('-' * 60)
print(f'  Powerplay: WI {pp1_runs}/{pp1_wkts}  |  IND {pp2_runs}/{pp2_wkts}')
print(f'  Dot %:     WI {dp1["Total"]}%  |  IND {dp2["Total"]}%')
print(f'  Boundary%: WI {bp1}%  |  IND {bp2}%')
print('-' * 60)
print(f'  WI Top bat:  {fmt_name(bs1[0]["Batsman"])} {bs1[0]["R"]}({bs1[0]["B"]}) SR {bs1[0]["SR"]}')
print(f'  IND Top bat: {fmt_name(bs2[0]["Batsman"])} {bs2[0]["R"]}({bs2[0]["B"]}) SR {bs2[0]["SR"]}')
bwl1_top = sorted(bw1, key=lambda x: (-x['W'], x['Econ']))[0]
bwl2_top = sorted(bw2, key=lambda x: (-x['W'], x['Econ']))[0]
print(f'  WI Top bowl: {fmt_name(bwl1_top["Bowler"])} {bwl1_top["W"]}W/{bwl1_top["O"]} Econ {bwl1_top["Econ"]}')
print(f'  IND Top bowl:{fmt_name(bwl2_top["Bowler"])} {bwl2_top["W"]}W/{bwl2_top["O"]} Econ {bwl2_top["Econ"]}')
print('=' * 60)

---
## 22. Generate Full Analysis PDF

The cells below contain the complete ReportLab PDF generation code embedded
directly (no subprocess).  Run them top-to-bottom after the inline charts above.

> **Expected output:** `PDF saved: C:/Users/i62476/Downloads/git_lxr_oem/IndiaVsWI_2ndODI_analysis.pdf`

In [ ]:
# ── ReportLab imports ─────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')   # switch to non-interactive backend for PDF charts
import matplotlib.pyplot as plt   # re-import so Agg is active
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                 TableStyle, Image, PageBreak)

# ── ReportLab colour palette ──────────────────────────────────────────────────
WI_MAROON  = colors.HexColor('#7B0C42')
WI_MPL     = '#7B0C42'
IND_BLUE   = colors.HexColor('#003C8F')
IND_MPL    = '#003C8F'
IND_TEXT   = colors.HexColor('#4488FF')
GRAY_LIGHT = colors.HexColor('#F2F2F2')
GRAY_MID   = colors.HexColor('#CCCCCC')
WHITE      = colors.white
BLACK      = colors.black
GREEN_GOOD = colors.HexColor('#27AE60')
RED_BAD    = colors.HexColor('#E74C3C')
ORANGE_MID = colors.HexColor('#E67E22')
BG_PAGE    = colors.HexColor('#0D0D1F')
DARK_BG    = '#0D0D1F'
PANEL_BG   = '#16213E'

W, H = A4

_t1c = WI_MPL
_t2c = IND_MPL

print('ReportLab imports and palette OK.')

In [ ]:
# ── Paragraph style factory ──────────────────────────────────────────────────
def PS(name, **kw):
    return ParagraphStyle(name, **kw)

_cell      = PS('_cell',      fontSize=9,  fontName='Helvetica',      leading=12,
                alignment=TA_LEFT,   textColor=BLACK)
_cell_c    = PS('_cell_c',    fontSize=9,  fontName='Helvetica',      leading=12,
                alignment=TA_CENTER, textColor=BLACK)
_cell_bold = PS('_cell_bold', fontSize=9,  fontName='Helvetica-Bold', leading=12,
                alignment=TA_LEFT,   textColor=BLACK)
_hdr_cell  = PS('_hdr_cell',  fontSize=9,  fontName='Helvetica-Bold', leading=12,
                alignment=TA_CENTER, textColor=WHITE)

def P(text, style=None, **kw):
    s = style or _cell
    if kw:
        s = PS(s.name+'_', parent=s, **kw)
    return Paragraph(str(text), s)

def PC(text, **kw):
    return P(text, _cell_c, **kw)

def section_hdr(title, bg=None):
    if bg is None:
        bg = IND_BLUE
    t = Table([[P(title, PS('sh', fontSize=13, fontName='Helvetica-Bold',
                             textColor=WHITE, alignment=TA_LEFT, leading=16))]],
              colWidths=[16.4*cm])
    t.setStyle(TableStyle([
        ('BACKGROUND',   (0,0),(-1,-1), bg),
        ('LEFTPADDING',  (0,0),(-1,-1), 10),
        ('RIGHTPADDING', (0,0),(-1,-1), 10),
        ('TOPPADDING',   (0,0),(-1,-1),  6),
        ('BOTTOMPADDING',(0,0),(-1,-1),  6),
    ]))
    return t

def mini_hdr(title, color=None):
    if color is None:
        color = GRAY_MID
    return Paragraph(f'<b>{title}</b>',
        PS('mh', fontSize=11, fontName='Helvetica-Bold',
           textColor=color, spaceBefore=8, spaceAfter=4))

def make_table(hdr_texts, data_rows, col_widths, hdr_bg=None, stripe=True):
    if hdr_bg is None:
        hdr_bg = IND_BLUE
    hdr_style = PS('th', fontSize=9, fontName='Helvetica-Bold',
                   textColor=WHITE, alignment=TA_CENTER, leading=12)
    hdrs = [Paragraph(h, hdr_style) for h in hdr_texts]
    rows = [hdrs] + data_rows
    t = Table(rows, colWidths=col_widths, repeatRows=1)
    cmds = [
        ('BACKGROUND',    (0,0),(-1,0), hdr_bg),
        ('ALIGN',         (0,0),(-1,-1),'CENTER'),
        ('VALIGN',        (0,0),(-1,-1),'MIDDLE'),
        ('GRID',          (0,0),(-1,-1), 0.4, GRAY_MID),
        ('BOTTOMPADDING', (0,0),(-1,-1), 5),
        ('TOPPADDING',    (0,0),(-1,-1), 5),
        ('LEFTPADDING',   (0,0),(-1,-1), 5),
        ('RIGHTPADDING',  (0,0),(-1,-1), 5),
    ]
    if stripe:
        cmds.append(('ROWBACKGROUNDS',(0,1),(-1,-1),[GRAY_LIGHT,WHITE]))
    t.setStyle(TableStyle(cmds))
    return t

def phase_table(ps, hdr_bg):
    cws = [3.2*cm,2.1*cm,2.1*cm,2.4*cm,2.4*cm,2.2*cm]
    rows = [[PC(p['Phase']), PC(p['Runs']), PC(p['Wkts']),
             PC(p['Overs']), PC(p['RR']),  PC(p['Dot%'])] for p in ps]
    return make_table(['Phase','Runs','Wkts','Overs','RR','Dot%'],
                      rows, cws, hdr_bg)

def bat_table(bs, hdr_bg, n=12):
    cws = [4.2*cm,1.4*cm,1.4*cm,1.4*cm,1.4*cm,2.0*cm,2.0*cm]
    rows = [[P(fmt_name(b['Batsman'])), PC(b['R']), PC(b['B']),
             PC(b['4s']), PC(b['6s']), PC(b['SR']), PC(f"{b['Dot%']}%")]
            for b in bs[:n]]
    return make_table(['Batsman','R','B','4s','6s','SR','Dot%'],
                      rows, cws, hdr_bg)

def bowl_table(bw, hdr_bg, n=10):
    cws = [4.2*cm,1.6*cm,1.6*cm,1.6*cm,2.2*cm,2.2*cm,1.5*cm,1.5*cm]
    rows = [[P(fmt_name(b['Bowler'])), PC(b['O']), PC(b['R']),
             PC(b['W']), PC(b['Econ']), PC(f"{b['Dot%']}%"),
             PC(b['Wd']), PC(b['NB'])]
            for b in bw[:n]]
    return make_table(['Bowler','O','R','W','Econ','Dot%','Wd','NB'],
                      rows, cws, hdr_bg)

print('Style helpers defined.')

In [ ]:
# ── fig_to_img: saves a matplotlib figure to BytesIO -> ReportLab Image ──────
def fig_to_img(fig, w_cm=16, h_cm=None):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    buf.seek(0)
    plt.close(fig)
    ratio = fig.get_figheight()/fig.get_figwidth()
    ww = w_cm*cm
    hh = h_cm*cm if h_cm else ww*ratio
    return Image(buf, width=ww, height=hh)

def chart_scoring():
    target = int(total1) + 1
    fig, ax = plt.subplots(figsize=(12,4.5))
    fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(PANEL_BG)
    x1=[p[0] for p in cum1]; y1=[p[1] for p in cum1]
    x2=[p[0] for p in cum2]; y2=[p[1] for p in cum2]
    ax.plot(x1,y1,color=_t1c,lw=2.5,label=f'{team1} ({total1}/{int(wkts1)})')
    ax.plot(x2,y2,color=_t2c,lw=2.5,label=f'{team2} ({total2}/{int(wkts2)})')
    ax.plot([0, 50], [0, target], color='#FFD700', lw=1.3, ls='--', alpha=0.65,
            label=f'Par (target {target})')
    for _, row in inn1[inn1['is_wicket']==1].iterrows():
        lb  = inn1.loc[:row.name,'legal'].sum()
        run = inn1.loc[:row.name,'team_run'].sum()
        ax.scatter(lb/6, run, color=_t1c, s=70, zorder=5, marker='v')
    for _, row in inn2[inn2['is_wicket']==1].iterrows():
        lb  = inn2.loc[:row.name,'legal'].sum()
        run = inn2.loc[:row.name,'team_run'].sum()
        ax.scatter(lb/6, run, color=_t2c, s=70, zorder=5, marker='v')
    ax.set_xlim(0,50); ax.set_ylim(0)
    ax.set_xlabel('Overs',color='white',fontsize=10)
    ax.set_ylabel('Runs',color='white',fontsize=10)
    ax.set_title('Scoring Progression  (\u25bc = wicket)',color='white',fontsize=13,fontweight='bold')
    ax.tick_params(colors='white'); ax.grid(alpha=0.2,color='white')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['bottom','left']].set_color('#555')
    ax.legend(facecolor=DARK_BG,labelcolor='white',fontsize=9)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 16)

def chart_rr():
    overs = list(range(1,51))
    rr1v  = [rr1.get(o,0) for o in overs]
    rr2v  = [rr2.get(o,0) for o in overs]
    fig, ax = plt.subplots(figsize=(12,4))
    fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(PANEL_BG)
    ax.plot(overs,rr1v,color=_t1c,lw=2.5,marker='o',ms=3,label=team1)
    ax.plot(overs,rr2v,color=_t2c,lw=2.5,marker='s',ms=3,label=team2)
    ax.axhline(6.0,color='white',lw=1,ls='--',alpha=0.35,label='6 RPO')
    ax.set_xlim(1,50); ax.set_ylim(0)
    ax.set_xlabel('Over',color='white',fontsize=10)
    ax.set_ylabel('Run Rate',color='white',fontsize=10)
    ax.set_title('Cumulative Run Rate by Over',color='white',fontsize=13,fontweight='bold')
    ax.tick_params(colors='white'); ax.grid(alpha=0.2,color='white')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['bottom','left']].set_color('#555')
    ax.legend(facecolor=DARK_BG,labelcolor='white',fontsize=9)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 16)

def chart_per_over():
    overs = list(range(1,51))
    r1=[inn1[inn1['over_no']==o]['team_run'].sum() for o in overs]
    r2=[inn2[inn2['over_no']==o]['team_run'].sum() for o in overs]
    w1=[inn1[inn1['over_no']==o]['is_wicket'].sum() for o in overs]
    w2=[inn2[inn2['over_no']==o]['is_wicket'].sum() for o in overs]
    fig,(ax1,ax2) = plt.subplots(1,2,figsize=(13,4))
    fig.patch.set_facecolor(DARK_BG)
    for ax,runs,wkts,color,label in [
        (ax1,r1,w1,_t1c,team1),(ax2,r2,w2,_t2c,team2)]:
        ax.set_facecolor(PANEL_BG)
        bars = ax.bar(overs,runs,color=color,alpha=0.85,zorder=3)
        for bar,w in zip(bars,wkts):
            if w>0:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                        'W'*int(w), ha='center', va='bottom',
                        color='#FF6B6B', fontsize=7, fontweight='bold')
        ax.set_xlabel('Over',color='white',fontsize=9)
        ax.set_ylabel('Runs',color='white',fontsize=9)
        ax.set_title(f'{label} \u2014 Runs per Over',color='white',fontsize=10,fontweight='bold')
        ax.tick_params(colors='white'); ax.set_xticks(list(range(5,51,5)))
        ax.set_xticklabels(list(range(5,51,5)),fontsize=7)
        ax.spines[['top','right']].set_visible(False)
        ax.spines[['bottom','left']].set_color('#555')
        ax.grid(axis='y',alpha=0.2,color='white',zorder=0)
        ax.set_xlim(0.5,50.5)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 16)

def chart_phase():
    phases=['Powerplay','Middle','Death']
    r1d={p['Phase']:p['Runs'] for p in ps1}
    r2d={p['Phase']:p['Runs'] for p in ps2}
    runs1=[r1d.get(p,0) for p in phases]
    runs2=[r2d.get(p,0) for p in phases]
    x=np.arange(len(phases)); w=0.35
    fig,ax = plt.subplots(figsize=(8,4))
    fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(PANEL_BG)
    b1=ax.bar(x-w/2,runs1,w,label=team1,color=_t1c,alpha=0.9,zorder=3)
    b2=ax.bar(x+w/2,runs2,w,label=team2,color=_t2c,alpha=0.9,zorder=3)
    for bar in list(b1)+list(b2):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                str(int(bar.get_height())), ha='center', va='bottom',
                color='white', fontsize=10, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(phases,color='white',fontsize=11)
    ax.set_ylabel('Runs',color='white',fontsize=10)
    ax.set_title('Phase-wise Runs',color='white',fontsize=12,fontweight='bold')
    ax.tick_params(colors='white')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['bottom','left']].set_color('#555')
    ax.grid(axis='y',alpha=0.2,color='white',zorder=0)
    ax.legend(facecolor=DARK_BG,labelcolor='white',fontsize=9)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 10)

def chart_dot_comparison():
    phases = ['Powerplay', 'Middle', 'Death', 'Total']
    x = np.arange(len(phases)); w = 0.35
    fig, ax = plt.subplots(figsize=(9, 4))
    fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(PANEL_BG)
    b1 = ax.bar(x-w/2, [dp1[p] for p in phases], w, label=_t1l, color=_t1c, alpha=0.9, zorder=3)
    b2 = ax.bar(x+w/2, [dp2[p] for p in phases], w, label=_t2l, color=_t2c, alpha=0.9, zorder=3)
    for bar in list(b1)+list(b2):
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.5, f'{h:.0f}%',
                ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(phases, color='white', fontsize=10)
    ax.set_ylabel('Dot Ball %', color='white', fontsize=10)
    ax.set_title('Dot Ball % by Phase', color='white', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 80)
    ax.tick_params(colors='white')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['bottom','left']].set_color('#555')
    ax.grid(axis='y', alpha=0.2, color='white', zorder=0)
    ax.legend(facecolor=DARK_BG, labelcolor='white', fontsize=9)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 10)

def chart_batsmen():
    top1=bs1[:6]; top2=bs2[:6]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(13,4.5))
    fig.patch.set_facecolor(DARK_BG)
    for ax,rows,color,label in [
        (ax1,top1,_t1c,team1),(ax2,top2,_t2c,team2)]:
        ax.set_facecolor(PANEL_BG)
        names=[fmt_name(r['Batsman']) for r in rows]
        runs =[r['R'] for r in rows]
        balls=[r['B'] for r in rows]
        y=np.arange(len(names))
        ax.barh(y+0.2,runs, 0.35,color=color,  alpha=0.9,label='Runs')
        ax.barh(y-0.2,balls,0.35,color='#888',alpha=0.7,label='Balls')
        for i,(r,b) in enumerate(zip(runs,balls)):
            ax.text(r+0.5,i+0.2,str(r), va='center',color='white',fontsize=9,fontweight='bold')
            ax.text(b+0.5,i-0.2,str(b), va='center',color='#ccc', fontsize=8)
        ax.set_yticks(y); ax.set_yticklabels(names,color='white',fontsize=9)
        ax.set_xlabel('Count',color='white',fontsize=9)
        ax.set_title(f'{label} \u2014 Top Batsmen',color='white',fontsize=10,fontweight='bold')
        ax.tick_params(colors='white')
        ax.spines[['top','right']].set_visible(False)
        ax.spines[['bottom','left']].set_color('#555')
        ax.grid(axis='x',alpha=0.2,color='white')
        ax.legend(facecolor=DARK_BG,labelcolor='white',fontsize=8)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 16)

def chart_bowling():
    top1=sorted(bw1,key=lambda x:-x['W'])[:6]
    top2=sorted(bw2,key=lambda x:-x['W'])[:6]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(13,4.5))
    fig.patch.set_facecolor(DARK_BG)
    for ax,rows,color,label in [
        (ax1,top1,_t1c, f'{team1} Bowlers (vs {team2})'),
        (ax2,top2,_t2c, f'{team2} Bowlers (vs {team1})')]:
        ax.set_facecolor(PANEL_BG)
        names=[fmt_name(r['Bowler']) for r in rows]
        wkts =[r['W']    for r in rows]
        econ =[r['Econ'] for r in rows]
        y=np.arange(len(names))
        ax.bar(y-0.2,wkts,0.35,color=color,alpha=0.9,label='Wickets')
        ax.bar(y+0.2,econ,0.35,color='#888',alpha=0.7,label='Economy')
        for i,(ww,e) in enumerate(zip(wkts,econ)):
            ax.text(i-0.2,ww+0.05,str(ww),ha='center',va='bottom',color='white',fontsize=9,fontweight='bold')
            ax.text(i+0.2,e+0.05, f'{e:.1f}',ha='center',va='bottom',color='#ccc', fontsize=8)
        ax.set_xticks(y); ax.set_xticklabels(names,color='white',fontsize=9)
        ax.set_title(label,color='white',fontsize=10,fontweight='bold')
        ax.tick_params(colors='white')
        ax.spines[['top','right']].set_visible(False)
        ax.spines[['bottom','left']].set_color('#555')
        ax.grid(axis='y',alpha=0.2,color='white')
        ax.legend(facecolor=DARK_BG,labelcolor='white',fontsize=8)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 16)

def chart_eco_vs_wkt():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor(DARK_BG)
    for ax, bw, color, label in [
        (ax1, bw1, _t1c, f'{team1} Bowlers'),
        (ax2, bw2, _t2c, f'{team2} Bowlers')]:
        ax.set_facecolor(PANEL_BG)
        wkts = [b['W'] for b in bw]
        econ = [b['Econ'] for b in bw]
        names = [fmt_name(b['Bowler']) for b in bw]
        ax.scatter(wkts, econ, c=color, s=120, alpha=0.85, zorder=3, edgecolors='white', linewidths=0.5)
        for w, e, n in zip(wkts, econ, names):
            ax.annotate(n, (w, e), textcoords='offset points', xytext=(5, 4),
                        color='white', fontsize=8)
        ax.axhline(6.0, color='#888', lw=1, ls='--', alpha=0.5, label='6 RPO')
        ax.set_xlabel('Wickets', color='white', fontsize=9)
        ax.set_ylabel('Economy Rate', color='white', fontsize=9)
        ax.set_title(f'{label}\nEconomy vs Wickets', color='white', fontsize=10, fontweight='bold')
        ax.tick_params(colors='white')
        ax.spines[['top','right']].set_visible(False)
        ax.spines[['bottom','left']].set_color('#555')
        ax.grid(alpha=0.15, color='white')
        if wkts:
            ax.set_xticks(range(0, max(wkts)+2))
        ax.legend(facecolor=DARK_BG, labelcolor='white', fontsize=8)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 16)

def chart_fow_partnerships():
    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor(DARK_BG)
    gs = fig.add_gridspec(2, 2, hspace=0.5, wspace=0.35)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])
    for ax in [ax1, ax2, ax3, ax4]:
        ax.set_facecolor(PANEL_BG)
        ax.tick_params(colors='white')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444466')

    def _plot_fow(ax, fow, color, title):
        if not fow:
            ax.text(0.5, 0.5, 'All Out / No Data', ha='center', va='center',
                    color='white', transform=ax.transAxes, fontsize=10)
            ax.set_title(title, color='white', fontsize=10, fontweight='bold'); return
        scores = [f['Score'] for f in fow]
        labels = ['W' + str(f['Wkt']) + ': ' + f['Batsman'] for f in fow]
        overs  = [f['Over'] for f in fow]
        y = range(len(fow))
        ax.barh(list(y), scores, color=color, alpha=0.8, height=0.6)
        max_s = max(scores) if scores else 1
        for i, (s, ov) in enumerate(zip(scores, overs)):
            ax.text(s + max_s*0.02, i, f'{s} (ov {ov})', va='center',
                    color='white', fontsize=7)
        ax.set_yticks(list(y)); ax.set_yticklabels(labels, color='white', fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel('Runs at Dismissal', color='white', fontsize=9)
        ax.set_title(title, color='white', fontsize=10, fontweight='bold', pad=4)

    def _plot_parts(ax, parts, color, title):
        if not parts:
            ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                    color='white', transform=ax.transAxes, fontsize=10)
            ax.set_title(title, color='white', fontsize=10, fontweight='bold'); return
        top = parts[:6]
        labels = ['P' + str(p['Wkt']) + ': ' + p['Bat1'] + ' & ' + p['Bat2'] for p in top]
        runs   = [p['Runs'] for p in top]
        y = range(len(top))
        ax.barh(list(y), runs, color=color, alpha=0.8, height=0.6)
        max_r = max(runs) if runs else 1
        for i, (r, p) in enumerate(zip(runs, top)):
            ax.text(r + max_r*0.02, i,
                    str(r) + ' (' + str(p['Balls']) + 'b, RR ' + str(p['RR']) + ')',
                    va='center', color='white', fontsize=7)
        ax.set_yticks(list(y)); ax.set_yticklabels(labels, color='white', fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel('Partnership Runs', color='white', fontsize=9)
        ax.set_title(title, color='white', fontsize=10, fontweight='bold', pad=4)

    _plot_fow(ax1, fow1, _t1c, f'{_t1l} \u2014 Fall of Wickets')
    _plot_fow(ax2, fow2, _t2c, f'{_t2l} \u2014 Fall of Wickets')
    _plot_parts(ax3, par1, _t1c, f'{_t1l} \u2014 Top Partnerships')
    _plot_parts(ax4, par2, _t2c, f'{_t2l} \u2014 Top Partnerships')
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, w_cm=17)

def chart_analytics():
    fig = plt.figure(figsize=(18, 12))
    fig.patch.set_facecolor(DARK_BG)
    gs = fig.add_gridspec(2, 2, hspace=0.55, wspace=0.38)
    ax_pp = fig.add_subplot(gs[0, 0])
    ax_bp = fig.add_subplot(gs[0, 1])
    ax_df = fig.add_subplot(gs[1, 0])
    ax_wk = fig.add_subplot(gs[1, 1])
    for ax in [ax_pp, ax_bp, ax_df, ax_wk]:
        ax.set_facecolor(PANEL_BG)
        ax.tick_params(colors='white')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444466')

    bars = ax_pp.bar([0, 1], [pp1_runs, pp2_runs], color=[_t1c, _t2c], width=0.5)
    for bar, wk in zip(bars, [pp1_wkts, pp2_wkts]):
        ax_pp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                   f'{int(bar.get_height())}/{wk}',
                   ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
    ax_pp.set_xticks([0, 1]); ax_pp.set_xticklabels([_t1l, _t2l], color='white', fontsize=12)
    ax_pp.set_title('Powerplay Battle (Ov 1-10)\nRuns / Wickets', color='white', fontsize=11, pad=8)
    ax_pp.set_ylabel('Runs', color='white', fontsize=10)
    ax_pp.set_ylim(0, max(pp1_runs, pp2_runs) * 1.3 + 5)

    bp_min = max(0, min(bp1, bp2) * 0.75)
    bp_max = min(100, max(bp1, bp2) * 1.3)
    bars_bp = ax_bp.bar([0, 1], [bp1, bp2], color=[_t1c, _t2c], width=0.5)
    for bar, val in zip(bars_bp, [bp1, bp2]):
        ax_bp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                   f'{val}%', ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
    ax_bp.set_xticks([0, 1]); ax_bp.set_xticklabels([_t1l, _t2l], color='white', fontsize=12)
    ax_bp.set_title('Boundary Contribution\n% of Total Runs from 4s & 6s', color='white', fontsize=11, pad=8)
    ax_bp.set_ylabel('%', color='white', fontsize=10)
    ax_bp.set_ylim(bp_min, bp_max)

    all_df_rows = ([(r['striker'], r['sr'], r['runs'], r['balls'], _t1c) for _, r in dbr1.iterrows()] +
                   [(r['striker'], r['sr'], r['runs'], r['balls'], _t2c) for _, r in dbr2.iterrows()])
    all_df_rows.sort(key=lambda x: x[1], reverse=True); all_df_rows = all_df_rows[:8]
    if all_df_rows:
        names_df = [fmt_name(x[0]) for x in all_df_rows]
        y_df = range(len(all_df_rows))
        bars_df = ax_df.barh(list(y_df), [x[1] for x in all_df_rows],
                             color=[x[4] for x in all_df_rows], height=0.6)
        max_sr = max(x[1] for x in all_df_rows) if all_df_rows else 100
        ax_df.set_xlim(0, max_sr * 1.28)
        for bar, rd in zip(bars_df, all_df_rows):
            ax_df.text(bar.get_width() + max_sr*0.02, bar.get_y() + bar.get_height()/2,
                       f'{int(rd[2])}({int(rd[3])})', va='center', ha='left',
                       color='white', fontsize=8)
        ax_df.set_yticks(list(y_df)); ax_df.set_yticklabels(names_df, color='white', fontsize=10)
        ax_df.invert_yaxis()
    ax_df.set_title('Death Overs Finishers (Ov 41-50)\nStrike Rate  (min 6 balls)', color='white', fontsize=11, pad=8)
    ax_df.set_xlabel('Strike Rate', color='white', fontsize=10)
    ax_df.axvline(100, color='#888888', linestyle='--', linewidth=0.8)

    all_wk_rows = ([(r['bowler'], r['wickets'], _t1c, _t1l) for _, r in wtr1.iterrows()] +
                   [(r['bowler'], r['wickets'], _t2c, _t2l) for _, r in wtr2.iterrows()])
    all_wk_rows.sort(key=lambda x: x[1], reverse=True); all_wk_rows = all_wk_rows[:10]
    if all_wk_rows:
        names_wk = [fmt_name(x[0]) for x in all_wk_rows]
        y_wk = range(len(all_wk_rows))
        bars_wk = ax_wk.barh(list(y_wk), [x[1] for x in all_wk_rows],
                             color=[x[2] for x in all_wk_rows], height=0.6)
        for bar, rd in zip(bars_wk, all_wk_rows):
            ax_wk.text(bar.get_width() + 0.06, bar.get_y() + bar.get_height()/2,
                       f'({rd[3]})', va='center', ha='left', color='white', fontsize=9, fontweight='bold')
        ax_wk.set_yticks(list(y_wk)); ax_wk.set_yticklabels(names_wk, color='white', fontsize=10)
        ax_wk.invert_yaxis()
        ax_wk.set_xticks(range(0, int(max(x[1] for x in all_wk_rows)) + 2))
    ax_wk.set_title('Wicket Distribution\nBowlers with Scalps', color='white', fontsize=11, pad=8)
    ax_wk.set_xlabel('Wickets', color='white', fontsize=10)
    return fig_to_img(fig, w_cm=17)

def chart_rolling_rr(window=5):
    overs = list(range(1, 51))
    def over_rr(inn):
        return {o: inn[inn['over_no']==o]['team_run'].sum() /
                   max(inn[inn['over_no']==o]['legal'].sum() / 6, 1e-9)
                for o in overs}
    rr1_by_ov = over_rr(inn1)
    rr2_by_ov = over_rr(inn2)
    def rolling(d, w):
        vals = [d.get(o, 0) for o in overs]
        return [sum(vals[max(0,i-w+1):i+1]) / len(vals[max(0,i-w+1):i+1]) for i in range(len(vals))]
    r1 = rolling(rr1_by_ov, window)
    r2 = rolling(rr2_by_ov, window)
    y_top = max(max(r1) if r1 else 0, max(r2) if r2 else 0) * 1.15 + 1
    fig, ax = plt.subplots(figsize=(14, 4))
    fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(PANEL_BG)
    ax.plot(overs, r1, color=_t1c, lw=2.5, marker='o', ms=3, label=f'{_t1l} ({window}-ov rolling RR)')
    ax.plot(overs, r2, color=_t2c, lw=2.5, marker='s', ms=3, label=f'{_t2l} ({window}-ov rolling RR)')
    ax.fill_between(overs, r1, alpha=0.12, color=_t1c)
    ax.fill_between(overs, r2, alpha=0.12, color=_t2c)
    for x_ph, lbl_ph in [(10.5, 'PP'), (40.5, 'Mid')]:
        ax.axvline(x_ph, color='#888', lw=1, ls=':', alpha=0.6)
        ax.text(x_ph + 0.15, y_top * 0.88, lbl_ph, color='#AAAACC', fontsize=8)
    ax.axhline(6.0, color='#FFD700', lw=1, ls='--', alpha=0.5, label='6 RPO')
    ax.set_xlim(1, 50); ax.set_ylim(0, y_top)
    ax.set_xlabel('Over', color='white', fontsize=10)
    ax.set_ylabel('Rolling RR', color='white', fontsize=10)
    ax.set_title(f'{window}-Over Rolling Run Rate  \u2014  Momentum Tracker',
                 color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white'); ax.grid(alpha=0.15, color='white')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['bottom','left']].set_color('#555')
    ax.legend(facecolor=DARK_BG, labelcolor='white', fontsize=9)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 16)

def chart_required_rate_tracker():
    target = int(total1) + 1
    if inn2.empty:
        return Spacer(1, 0.1*cm)
    cum = 0
    cum_scores = {0: 0}
    for o in sorted(inn2['over_no'].unique()):
        cum += inn2[inn2['over_no'] == o]['team_run'].sum()
        cum_scores[o] = int(cum)
    max_ov = max(k for k in cum_scores if k > 0)
    overs_plot = list(range(1, max_ov + 1))
    act_rr, req_rr = [], []
    for o in overs_plot:
        sc_keys  = [k for k in cum_scores if k <= o]
        sc       = cum_scores[max(sc_keys)] if sc_keys else 0
        prev_keys = [k for k in cum_scores if k < o]
        prev     = cum_scores[max(prev_keys)] if prev_keys else 0
        act_rr.append(sc / o)
        req_rr.append((target - prev) / max(51 - o, 1))
    fig, ax = plt.subplots(figsize=(14, 4))
    fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(PANEL_BG)
    ax.plot(overs_plot, act_rr, color=_t2c, lw=2.5, marker='o', ms=4, label=f'{_t2l} Actual RR')
    ax.plot(overs_plot, req_rr, color='#FFD700', lw=2, ls='--', marker='s', ms=3, label='Required RR')
    for i in range(len(overs_plot) - 1):
        c = '#27AE60' if act_rr[i] >= req_rr[i] else '#E74C3C'
        ax.fill_between(overs_plot[i:i+2], act_rr[i:i+2], req_rr[i:i+2], alpha=0.18, color=c)
    ax.set_xlim(1, max_ov); ax.set_ylim(0)
    ax.set_xlabel('Over', color='white', fontsize=10)
    ax.set_ylabel('Run Rate', color='white', fontsize=10)
    ax.set_title(f'{_t2l} Chase Analysis  \u2014  Actual vs Required RR  (target {target})',
                 color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white'); ax.grid(alpha=0.15, color='white')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['bottom','left']].set_color('#555')
    ax.legend(facecolor=DARK_BG, labelcolor='white', fontsize=9)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 16)

def chart_batsman_contribution():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor(DARK_BG)
    for ax, bs, inn, color, label in [(ax1, bs1, inn1, _t1c, _t1l), (ax2, bs2, inn2, _t2c, _t2l)]:
        ax.set_facecolor(PANEL_BG)
        t_total  = inn['team_run'].sum()
        extras   = int(t_total - inn['runs_off_bat'].sum())
        top      = bs[:7]
        names    = [fmt_name(b['Batsman']) for b in top]
        runs_v   = [b['R'] for b in top]
        other    = max(0, int(inn['runs_off_bat'].sum()) - sum(runs_v))
        all_runs  = runs_v + [other, extras]
        all_names = names + ['Others', 'Extras']
        bar_colors = [color] * len(top) + ['#778899', '#555566']
        pcts = [r / t_total * 100 if t_total > 0 else 0 for r in all_runs]
        y = range(len(all_names))
        bars = ax.barh(list(y), pcts, color=bar_colors, height=0.7, alpha=0.9)
        for bar, run, pct in zip(bars, all_runs, pcts):
            if pct > 1.5:
                ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                        f'{int(run)} ({pct:.0f}%)', va='center', ha='left', color='white', fontsize=8)
        ax.set_yticks(list(y)); ax.set_yticklabels(all_names, color='white', fontsize=9)
        ax.set_xlabel('% of Team Total', color='white', fontsize=9)
        ax.set_title(f'{label} \u2014 Batsman Contribution', color='white', fontsize=10, fontweight='bold')
        ax.invert_yaxis()
        ax.tick_params(colors='white')
        ax.spines[['top','right']].set_visible(False)
        ax.spines[['bottom','left']].set_color('#555')
        ax.grid(axis='x', alpha=0.15, color='white')
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 16)

def chart_boundary_by_phase():
    phases = ['Powerplay', 'Middle', 'Death']
    def bnd_by_phase(inn):
        return ({p: int(inn[inn['phase']==p]['is_4'].sum()) for p in phases},
                {p: int(inn[inn['phase']==p]['is_6'].sum()) for p in phases})
    f1, s1 = bnd_by_phase(inn1)
    f2, s2 = bnd_by_phase(inn2)
    x = np.arange(len(phases)); w = 0.2
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor(DARK_BG); ax.set_facecolor(PANEL_BG)
    b1 = ax.bar(x - 1.5*w, [f1[p] for p in phases], w, label=f'{_t1l} 4s', color=_t1c, alpha=0.9)
    b2 = ax.bar(x - 0.5*w, [s1[p] for p in phases], w, label=f'{_t1l} 6s', color=_t1c, alpha=0.6, hatch='//')
    b3 = ax.bar(x + 0.5*w, [f2[p] for p in phases], w, label=f'{_t2l} 4s', color=_t2c, alpha=0.9)
    b4 = ax.bar(x + 1.5*w, [s2[p] for p in phases], w, label=f'{_t2l} 6s', color=_t2c, alpha=0.6, hatch='//')
    for bar in list(b1) + list(b2) + list(b3) + list(b4):
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, str(int(h)),
                    ha='center', va='bottom', color='white', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(phases, color='white', fontsize=11)
    ax.set_ylabel('Count', color='white', fontsize=10)
    ax.set_title('Boundary Distribution by Phase  (4s solid, 6s hatched)',
                 color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['bottom','left']].set_color('#555')
    ax.grid(axis='y', alpha=0.15, color='white')
    ax.legend(facecolor=DARK_BG, labelcolor='white', fontsize=9, ncol=2)
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 14)

def chart_dismissal_types():
    wk1 = inn1[inn1['is_wicket']==1]['wicket_type'].dropna().value_counts()
    wk2 = inn2[inn2['is_wicket']==1]['wicket_type'].dropna().value_counts()
    if wk1.empty and wk2.empty:
        return Spacer(1, 0.1*cm)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    fig.patch.set_facecolor(DARK_BG)
    for ax, wk, color, label in [(ax1, wk1, _t1c, _t1l), (ax2, wk2, _t2c, _t2l)]:
        ax.set_facecolor(PANEL_BG)
        if wk.empty:
            ax.text(0.5, 0.5, 'No wickets', ha='center', va='center',
                    color='white', transform=ax.transAxes, fontsize=10)
            ax.set_title(f'{label} \u2014 Dismissal Types', color='white', fontsize=10, fontweight='bold')
            continue
        y = range(len(wk))
        bars = ax.barh(list(y), wk.values, color=color, alpha=0.85, height=0.6)
        for bar, val in zip(bars, wk.values):
            ax.text(bar.get_width() + 0.06, bar.get_y() + bar.get_height()/2,
                    str(val), va='center', ha='left', color='white', fontsize=9)
        ax.set_yticks(list(y)); ax.set_yticklabels(wk.index.tolist(), color='white', fontsize=9)
        ax.set_xlabel('Count', color='white', fontsize=9)
        ax.set_title(f'{label} \u2014 Dismissal Types', color='white', fontsize=10, fontweight='bold')
        ax.invert_yaxis()
        ax.tick_params(colors='white')
        ax.spines[['top','right']].set_visible(False)
        ax.spines[['bottom','left']].set_color('#555')
        ax.grid(axis='x', alpha=0.15, color='white')
    plt.tight_layout(pad=0.5)
    return fig_to_img(fig, 15)

print('fig_to_img and all chart functions defined.')

In [ ]:
def make_match_context():
    toss_desc = f'{toss_winner} elected to {toss_decision}'
    if winner_wkts:
        result_mode = f'{winner} won by {winner_wkts} wickets (chasing)'
    elif winner_runs:
        result_mode = f'{winner} won by {winner_runs} runs (defending)'
    else:
        result_mode = f'{winner} won'
    pp_adv = _t1l if pp1_runs > pp2_runs else _t2l
    pp_note = f'{pp_adv} led powerplay by +{abs(pp1_runs - pp2_runs)} runs ({pp1_runs}/{pp1_wkts} vs {pp2_runs}/{pp2_wkts})'
    bp_adv  = _t1l if bp1 > bp2 else _t2l
    bp_note = f'{bp_adv} hit {abs(round(bp1 - bp2, 1))}% more boundary runs ({bp1}% vs {bp2}%)'
    if cl1_wk >= cl2_wk and cl1_wk > 0:
        mo_note = f'{_t2l} took {cl1_wk}-wicket cluster from over {cl1_ov}; {_t1l} conceded {cl1_ru} runs in that span'
    elif cl2_wk > 0:
        mo_note = f'{_t1l} took {cl2_wk}-wicket cluster from over {cl2_ov}; {_t2l} conceded {cl2_ru} runs in that span'
    else:
        mo_note = 'No clear wicket cluster \u2014 pressure maintained evenly'
    rows = [['Toss', toss_desc], ['Result Mode', result_mode],
            ['Powerplay Edge', pp_note], ['Boundary Edge', bp_note], ['Momentum Shift', mo_note]]
    t = Table(rows, colWidths=[3.5*cm, 13.0*cm])
    t.setStyle(TableStyle([
        ('ROWBACKGROUNDS', (0,0), (-1,-1), [colors.HexColor('#0D1033'), colors.HexColor('#0A0A25')]),
        ('TEXTCOLOR',    (0,0), (0,-1), colors.HexColor('#90CAF9')),
        ('TEXTCOLOR',    (1,0), (1,-1), colors.white),
        ('FONTNAME',     (0,0), (0,-1), 'Helvetica-Bold'),
        ('FONTSIZE',     (0,0), (-1,-1), 9),
        ('GRID',         (0,0), (-1,-1), 0.5, colors.HexColor('#333355')),
        ('LEFTPADDING',  (0,0), (-1,-1), 8),
        ('RIGHTPADDING', (0,0), (-1,-1), 8),
        ('TOPPADDING',   (0,0), (-1,-1), 5),
        ('BOTTOMPADDING',(0,0), (-1,-1), 5),
    ]))
    return t

def make_match_snapshot():
    top_bat1 = bs1[0] if bs1 else {'Batsman':'N/A','R':0,'B':0,'SR':0}
    top_bat2 = bs2[0] if bs2 else {'Batsman':'N/A','R':0,'B':0,'SR':0}
    top_bwl1 = sorted(bw1, key=lambda x: (-x['W'], x['Econ']))[0] if bw1 else {'Bowler':'N/A','W':0,'O':'0','Econ':0}
    top_bwl2 = sorted(bw2, key=lambda x: (-x['W'], x['Econ']))[0] if bw2 else {'Bowler':'N/A','W':0,'O':'0','Econ':0}
    f4_1 = int(inn1['is_4'].sum()); s6_1 = int(inn1['is_6'].sum())
    f4_2 = int(inn2['is_4'].sum()); s6_2 = int(inn2['is_6'].sum())
    wd1  = int(inn1['wides'].sum()); nb1 = int(inn1['noballs'].sum())
    wd2  = int(inn2['wides'].sum()); nb2 = int(inn2['noballs'].sum())
    snap_data = [
        ('Score',        f'{total1}/{int(wkts1)} in {ov1_str} ov',  f'{total2}/{int(wkts2)} in {ov2_str} ov'),
        ('Run Rate',     f'{total1/(legal1/6):.2f} RPO',            f'{total2/(legal2/6):.2f} RPO'),
        ('Powerplay',    f'{pp1_runs}/{pp1_wkts}  (RR {pp1_runs/10:.1f})',  f'{pp2_runs}/{pp2_wkts}  (RR {pp2_runs/10:.1f})'),
        ('Boundaries',   f'{f4_1}x4  +  {s6_1}x6',                 f'{f4_2}x4  +  {s6_2}x6'),
        ('Dot Ball %',   f'{dp1["Total"]:.0f}%',                     f'{dp2["Total"]:.0f}%'),
        ('Extras',       f'Wides: {wd1}   NB: {nb1}',               f'Wides: {wd2}   NB: {nb2}'),
        ('Top Batter',   f'{fmt_name(top_bat1["Batsman"])}  {top_bat1["R"]}({top_bat1["B"]})  SR {top_bat1["SR"]}',
                         f'{fmt_name(top_bat2["Batsman"])}  {top_bat2["R"]}({top_bat2["B"]})  SR {top_bat2["SR"]}'),
        ('Top Bowler',   f'{fmt_name(top_bwl1["Bowler"])}  {top_bwl1["W"]}W  econ {top_bwl1["Econ"]}',
                         f'{fmt_name(top_bwl2["Bowler"])}  {top_bwl2["W"]}W  econ {top_bwl2["Econ"]}'),
        ('Toss',         f'{toss_winner} elected to {toss_decision}', ''),
        ('Result',       f'{winner} won' + (f' by {winner_wkts} wickets' if winner_wkts else (f' by {winner_runs} runs' if winner_runs else '')), ''),
    ]
    hs = PS('snap_h', fontSize=10, fontName='Helvetica-Bold', textColor=WHITE,
            alignment=TA_CENTER, leading=13)
    hdr = [Paragraph('Metric', hs), Paragraph(_t1l, hs), Paragraph(_t2l, hs)]
    cws = [3.2*cm, 6.6*cm, 6.6*cm]
    lbl_s = PS('snap_lbl', fontSize=9, fontName='Helvetica-Bold',
               textColor=colors.HexColor('#90CAF9'), alignment=TA_CENTER, leading=12)
    val_s = PS('snap_val', fontSize=9, fontName='Helvetica', textColor=WHITE,
               alignment=TA_CENTER, leading=12)
    data_rows = []
    for metric, v1, v2 in snap_data:
        data_rows.append([Paragraph(metric, lbl_s), Paragraph(v1, val_s), Paragraph(v2, val_s)])
    t = Table([hdr] + data_rows, colWidths=cws, repeatRows=1)
    t.setStyle(TableStyle([
        ('BACKGROUND',     (0,0), (-1,0), colors.HexColor('#1A237E')),
        ('BACKGROUND',     (0,1), (0,-1), colors.HexColor('#0D1033')),
        ('ROWBACKGROUNDS', (1,1), (-1,-1), [colors.HexColor('#0A0A25'), colors.HexColor('#12122E')]),
        ('ALIGN',          (0,0), (-1,-1), 'CENTER'),
        ('VALIGN',         (0,0), (-1,-1), 'MIDDLE'),
        ('GRID',           (0,0), (-1,-1), 0.5, colors.HexColor('#333355')),
        ('TOPPADDING',     (0,0), (-1,-1), 6),
        ('BOTTOMPADDING',  (0,0), (-1,-1), 6),
        ('LEFTPADDING',    (0,0), (-1,-1), 6),
        ('RIGHTPADDING',   (0,0), (-1,-1), 6),
    ]))
    return t

def make_over_comparison_table():
    def ov_data(inn):
        d = {}; cum = 0
        for o in range(1, 51):
            g = inn[inn['over_no']==o]
            if not g.empty:
                r = int(g['team_run'].sum()); cum += r; d[o] = (r, cum)
        return d
    d1 = ov_data(inn1)
    d2 = ov_data(inn2)
    PHASE_COLORS = {'Powerplay': colors.HexColor('#0D2B63'),
                    'Middle':    colors.HexColor('#0D4A1A'),
                    'Death':     colors.HexColor('#4A0D0D')}
    def phase_of(o):
        return 'Powerplay' if o <= 10 else ('Middle' if o <= 40 else 'Death')
    hs = PS('oct_h', fontSize=8, fontName='Helvetica-Bold', textColor=WHITE,
            alignment=TA_CENTER, leading=11)
    vs = PS('oct_v', fontSize=8, fontName='Helvetica', textColor=WHITE,
            alignment=TA_CENTER, leading=11)
    hdr = [Paragraph(h, hs) for h in
           ['Ov', f'{_t1l} Runs', f'{_t1l} Score', f'{_t2l} Runs', f'{_t2l} Score']]
    rows = [hdr]; cws = [1.5*cm, 3.2*cm, 3.2*cm, 3.2*cm, 3.2*cm]; phase_bgs = []
    for o in range(1, 51):
        ph = phase_of(o)
        if o in d1:
            r1_v, sc1 = d1[o]
        else:
            prev_d1 = [v for k, v in d1.items() if k < o]
            r1_v, sc1 = 0, (prev_d1[-1][1] if prev_d1 else 0)
        if o in d2:
            r2_v, sc2 = d2[o]; r2_txt = str(r2_v); sc2_txt = str(sc2)
        else:
            r2_txt = '\u2013'; sc2_txt = '\u2013'
        rows.append([Paragraph(str(o), vs), Paragraph(str(r1_v), vs),
                     Paragraph(str(sc1), vs), Paragraph(r2_txt, vs),
                     Paragraph(sc2_txt, vs)])
        phase_bgs.append(PHASE_COLORS[ph])
    t = Table(rows, colWidths=cws, repeatRows=1)
    cmds = [('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1A237E')),
            ('ALIGN', (0,0), (-1,-1), 'CENTER'), ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
            ('GRID', (0,0), (-1,-1), 0.4, colors.HexColor('#333355')),
            ('TOPPADDING', (0,0), (-1,-1), 4), ('BOTTOMPADDING', (0,0), (-1,-1), 4),
            ('LEFTPADDING', (0,0), (-1,-1), 5), ('RIGHTPADDING', (0,0), (-1,-1), 5)]
    for i, bg in enumerate(phase_bgs):
        cmds.append(('BACKGROUND', (0, i+1), (-1, i+1), bg))
    t.setStyle(TableStyle(cmds))
    return t

def make_partnership_table(par, hdr_bg, n=8):
    cws = [1.2*cm, 3.2*cm, 3.2*cm, 1.8*cm, 1.8*cm, 2.0*cm, 2.2*cm]
    rows = []
    for p in par[:n]:
        sr = round(p['Runs'] / p['Balls'] * 100, 1) if p['Balls'] > 0 else 0.0
        rows.append([PC(p['Wkt']), P(p['Bat1']), P(p['Bat2']),
                     PC(p['Runs']), PC(p['Balls']), PC(p['RR']), PC(sr)])
    return make_table(['Wkt','Bat1','Bat2','Runs','Balls','RR','SR'], rows, cws, hdr_bg)

print('Table builder functions defined.')

In [ ]:
def auto_shining():
    rows = []
    for bs, team in [(bs1, team1), (bs2, team2)]:
        for b in [x for x in bs if x['B'] >= 15][:2]:
            rows.append((fmt_name(b['Batsman']), team, f"{b['R']}({b['B']})", f"SR {b['SR']}",
                         f"{b['R']} runs off {b['B']} balls, SR {b['SR']}; "
                         f"{b['4s']}x4  {b['6s']}x6  {b['Dot%']}% dots"))
    for bw, team in [(bw1, team1), (bw2, team2)]:
        top = sorted([x for x in bw if x['W'] >= 1], key=lambda x: (-x['W'], x['Econ']))
        for b in top[:2]:
            rows.append((fmt_name(b['Bowler']), team, b['O'],
                         f"{b['W']}W / Econ {b['Econ']}",
                         f"{b['W']}W in {b['O']} overs, economy {b['Econ']}; "
                         f"{b['Dot%']}% dot balls"))
    return rows[:8]

def auto_pressure():
    rows = []
    for bs, team in [(bs1, team1), (bs2, team2)]:
        poor = sorted([x for x in bs if x['B'] >= 10 and x['SR'] < 80], key=lambda x: x['SR'])
        for b in poor[:2]:
            rows.append((fmt_name(b['Batsman']), team, f"{b['R']}({b['B']})", f"SR {b['SR']}",
                         f"{b['R']} from {b['B']} balls, SR {b['SR']}; "
                         f"{b['Dot%']}% dots \u2014 failed to accelerate when needed"))
    for bw, team in [(bw1, team1), (bw2, team2)]:
        expensive = sorted(
            [x for x in bw if int(x['O'].split('.')[0]) >= 3 and x['Econ'] > 7.0],
            key=lambda x: -x['Econ'])
        for b in expensive[:2]:
            rows.append((fmt_name(b['Bowler']), team, b['O'], f"Econ {b['Econ']}",
                         f"{b['R']} runs in {b['O']} overs at economy {b['Econ']} \u2014 "
                         f"far too expensive, gave momentum away"))
    return rows[:7]

def auto_turning_points():
    def inn_events(inn, bat_team, bowl_team):
        evts = {}
        cum_r = 0; cum_l = 0; cum_wk = 0
        for ov, g in inn.groupby('over_no'):
            ov_r  = int(g['team_run'].sum())
            ov_wk = int(g['is_wicket'].sum())
            cum_r += ov_r; cum_l += int(g['legal'].sum()); cum_wk += ov_wk
            score_str = f'{cum_r}/{cum_wk}'
            if ov_wk >= 2:
                dis = g[g['is_wicket']==1]['striker'].apply(fmt_name).tolist()
                evts[ov] = (f'Ov {ov}\n({bat_team})',
                            ' & '.join(dis) + ' dismissed',
                            score_str,
                            f'{ov_wk}W over \u2014 {bowl_team} seizes momentum', 3)
            elif ov_r >= 18 and ov not in evts:
                evts[ov] = (f'Ov {ov}\n({bat_team})',
                            f'{ov_r} runs scored',
                            score_str,
                            f'Biggest over for {bat_team} \u2014 attack launched on {bowl_team}', 2)
            elif ov <= 10 and cum_wk >= 2 and ov_wk >= 1 and ov not in evts:
                evts[ov] = (f'Ov {ov}\n({bat_team})',
                            f'{cum_wk}W in powerplay',
                            score_str,
                            f'{bat_team} powerplay under pressure \u2014 {cum_r} runs in {ov} overs', 1)
        top = sorted(evts.values(), key=lambda x: -x[4])[:4]
        return sorted(top, key=lambda x: int(x[0].split('\n')[0].replace('Ov ','')))

    e1 = inn_events(inn1, _t1l, _t2l)
    e2 = inn_events(inn2, _t2l, _t1l)
    rows = []
    for e in e1 + e2:
        rows.append([PC(e[0]), P(e[1]), PC(e[2]), P(e[3])])
    return rows if rows else [[PC('\u2013'), P('No key events'), PC('\u2013'), P('\u2013')]]

print('Auto-derived performer/turning-point functions defined.')

In [ ]:
def cover_canvas(c, doc):
    c.saveState()
    c.setFillColor(BG_PAGE)
    c.rect(0, 0, W, H, fill=1, stroke=0)
    mid_y = H * 0.52
    c.setFillColor(WI_MAROON)
    c.rect(0, mid_y, W*0.5, H-mid_y, fill=1, stroke=0)
    c.setFillColor(IND_BLUE)
    c.rect(W*0.5, mid_y, W*0.5, H-mid_y, fill=1, stroke=0)
    c.setFont('Helvetica-Bold', 30)
    c.setFillColor(colors.HexColor('#FFD700'))
    c.drawCentredString(W*0.25, mid_y + (H-mid_y)*0.45, 'WI')
    c.setFont('Helvetica', 12)
    c.setFillColor(WHITE)
    c.drawCentredString(W*0.25, mid_y + (H-mid_y)*0.3, team1)
    c.setFont('Helvetica-Bold', 30)
    c.setFillColor(colors.HexColor('#FF9933'))
    c.drawCentredString(W*0.75, mid_y + (H-mid_y)*0.45, 'IND')
    c.setFont('Helvetica', 12)
    c.setFillColor(WHITE)
    c.drawCentredString(W*0.75, mid_y + (H-mid_y)*0.3, team2)
    c.setFont('Helvetica-Bold', 22)
    c.setFillColor(WHITE)
    c.drawCentredString(W*0.5, mid_y + (H-mid_y)*0.4, 'VS')
    c.setStrokeColor(WHITE); c.setLineWidth(2)
    c.line(W*0.5, mid_y, W*0.5, H)
    cy = mid_y - 0.5*cm
    c.setFont('Helvetica-Bold', 11)
    c.setFillColor(colors.HexColor('#AAAACC'))
    c.drawCentredString(W/2, cy, 'MATCH ANALYSIS REPORT  |  2nd ODI  |  WI Tour of India 2026')
    cy -= 1.0*cm
    winner_abbr = 'WI' if winner == team1 else 'IND'
    if winner_wkts:
        result_text = f'{winner_abbr} won by {winner_wkts} wickets'
    elif winner_runs:
        result_text = f'{winner_abbr} won by {winner_runs} runs'
    else:
        result_text = f'{winner_abbr} won'
    c.setFont('Helvetica-Bold', 20)
    c.setFillColor(GREEN_GOOD)
    c.drawCentredString(W/2, cy, result_text)
    cy -= 0.9*cm
    box_w = 13*cm; box_h = 3.2*cm
    box_x = (W - box_w)/2
    box_y = cy - box_h
    c.setFillColor(colors.HexColor('#1A1A3E'))
    c.roundRect(box_x, box_y, box_w, box_h, 6, fill=1, stroke=0)
    c.setFont('Helvetica-Bold', 9)
    c.setFillColor(GRAY_MID)
    for i, hdr in enumerate(['Team','Score','Overs','RR']):
        xpos = box_x + i*box_w/4 + box_w/8
        c.drawCentredString(xpos, box_y + box_h - 0.55*cm, hdr)
    c.setFont('Helvetica-Bold', 11)
    c.setFillColor(colors.HexColor('#FFD700'))
    vals1 = ['WI', f'{total1}/{int(wkts1)}', ov1_str, f'{total1/(legal1/6):.2f}']
    for i, v in enumerate(vals1):
        xpos = box_x + i*box_w/4 + box_w/8
        c.drawCentredString(xpos, box_y + box_h - 1.3*cm, str(v))
    c.setFillColor(WHITE)
    vals2 = ['IND', f'{total2}/{int(wkts2)}', ov2_str, f'{total2/(legal2/6):.2f}']
    for i, v in enumerate(vals2):
        xpos = box_x + i*box_w/4 + box_w/8
        c.drawCentredString(xpos, box_y + box_h - 2.1*cm, str(v))
    cy = box_y - 0.7*cm
    c.setFont('Helvetica', 9)
    c.setFillColor(GRAY_MID)
    c.drawCentredString(W/2, cy, f'Match {match_no}  |  {match_date}  |  {venue}')
    cy -= 0.55*cm
    c.setFont('Helvetica-Bold', 10)
    c.setFillColor(colors.HexColor('#FFD700'))
    c.drawCentredString(W/2, cy, f'Player of the Match:  {pom}')
    c.restoreState()

def dark_bg_canvas(c, doc):
    c.saveState()
    c.setFillColor(BG_PAGE)
    c.rect(0, 0, W, H, fill=1, stroke=0)
    c.setFont('Helvetica', 8)
    c.setFillColor(colors.HexColor('#666688'))
    c.drawCentredString(W/2, 0.6*cm, f'Page {doc.page}')
    c.restoreState()

print('cover_canvas and dark_bg_canvas defined.')

In [ ]:
# ── Build PDF story and write file ───────────────────────────────────────────
# Inference box helper (local to this cell)
from reportlab.platypus import ListFlowable, ListItem as _LI
from reportlab.lib.styles import ParagraphStyle as _PS2
from reportlab.lib.enums import TA_LEFT as _TAL

def _bul(txt):
    return Paragraph(
        '\u2022\u2002' + txt,
        _PS2('ibul', fontSize=7.8, fontName='Helvetica', leading=10.5,
             textColor=colors.HexColor('#CCCCCC'), alignment=_TAL,
             leftIndent=8)
    )

def inf_box(*bullets):
    rows = [[_bul(b)] for b in bullets]
    t = Table(rows, colWidths=[16.4*cm])
    t.setStyle(TableStyle([
        ('BACKGROUND',    (0,0),(-1,-1), colors.HexColor('#0D1B3E')),
        ('LEFTPADDING',   (0,0),(-1,-1), 10),
        ('RIGHTPADDING',  (0,0),(-1,-1), 10),
        ('TOPPADDING',    (0,0),(-1,-1),  2.5),
        ('BOTTOMPADDING', (0,0),(-1,-1),  2.5),
        ('LINEAFTER',     (0,0),(0,-1),  3, colors.HexColor('#00BCD4')),
    ]))
    return t

INF = {
    'scoring': [
        "India's cumulative runs overtook WI's trajectory from ~over 22 onwards and widened steadily.",
        f"WI {int(total1)}/{int(wkts1)} was formidable but insufficient — India crossed the target in over {int(legal2//6)+1} ({300-int(legal2)} balls to spare).",
        "WI lost 4 wickets in death overs (41\u201350), halting what could have been a 430+ total.",
        "India lost only 2 wickets throughout — the chase was never under pressure.",
    ],
    'rr': [
        "WI's cumulative RR peaked at 9.60 (PP) but declined to ~8.10 by the 50th over.",
        "India's cumulative RR climbed to ~9.34 — scoring faster on average across the entire chase.",
        "India's middle overs (9.33 RPO) decisively outperformed WI's (7.77 RPO) for the same wicket cost.",
        "The two lines cross around over 25: from there India's scoring curve never looks back.",
    ],
    'perover': [
        "WI's peak over: Over 7 (22 runs), but inconsistency in death (4 wickets, 76 runs in 41\u201350).",
        "India's peak over: Over 29 (25 runs); overall per-over scoring was far more consistent.",
        "WI's bar chart shows multiple low-scoring death overs caused by late-innings collapses.",
        "India had no collapse over during the chase — their distribution is notably flatter.",
    ],
    'phase': [
        "Middle overs decisive: India 280/2 vs WI 233/2 — 47 extra runs for the same wicket cost.",
        "Powerplay near-equal (WI 96/1 vs IND 92/0) but India's 0 PP wickets gave a priceless platform.",
        "WI death collapse: 76/4 in overs 41\u201350 — cost them at least 30\u201340 runs on a true total.",
        "India won without needing a full death innings (batted only 3.3 overs in overs 41\u201344).",
    ],
    'dot': [
        "India's middle-over dot% (28.9%) = lowest of any team/phase — Gill & Rohit in supreme control.",
        "India's death dot% (19.0%) shows near-total freedom in the final 3.3 overs of the chase.",
        "India overall dot% 33.0% vs WI's 37.7% — India converted more scoring opportunities.",
        "WI's death dot% (43.3%) confirms India's bowling created real pressure in overs 41\u201350.",
    ],
    'bat': [
        "WI: 3 centurions — Jangoo 114(77) SR=148, Hope 104(94) SR=111, Campbell 101(68) SR=149.",
        "India: Gill 223*(133) SR=168 with 26 fours and 8 sixes — one of ODI's great chasing innings.",
        "Rohit 101(75) SR=135; their 255-run opening stand made the result inevitable by over 28.",
        "WI's 3 centuries still weren't enough — India's SR advantage was the decisive factor.",
    ],
    'fow': [
        "WI: Jangoo\u2013Hope 3rd-wicket partnership of 182(130) was the innings backbone.",
        "India: Rohit\u2013Gill opening stand of 255(157) — among the all-time great chasing partnerships.",
        "WI's death collapse (4 wickets in overs 41\u201350) fragmented what should have been a 420+ total.",
        "Gill featured in all 3 India partnerships — the entire chase revolved around him.",
    ],
    'bowl': [
        "India's best figures: Kuldeep Yadav 2/65 Econ 6.50; Ravindra Jadeja also took 2 wickets (2/75).",
        "Gurnoor Brar 2/96 Econ 9.60 — took wickets but cost India nearly 100 runs in 10 overs.",
        "WI's best economy: Roston Chase 7.41 — with ZERO wickets. The bowling attack was harmless.",
        "Alzarri Joseph 0/89 Econ 11.12 and Keemo Paul 0/49 Econ 16.33 highlight WI's powerlessness.",
    ],
    'eco': [
        "No bowler from either team occupied the ideal zone (high wickets + low economy).",
        "Kuldeep Yadav closest to productive: 2w at 6.50 Econ — the standout performer with ball.",
        "All WI bowlers: economy >7.0 and \u22641 wicket — clustered in the worst quadrant.",
        "Keemo Paul (0w, 16.33 Econ) is the match's most extreme outlier with ball.",
    ],
    'analytics': [
        "PP near-equal (WI 96/1 vs IND 92/0) but India's 0 PP wickets were the decisive difference.",
        "Boundary % both high: WI 55.3%, IND 62.6% — flat pitch, boundary-heavy contest.",
        "India: 41 fours + 15 sixes; WI: 38 fours + 12 sixes — India's superior hitting rate tells.",
        "WI wicket distribution: 4 of 7 in death overs — innings collapsed when it mattered most.",
    ],
    'rolling': [
        "WI's rolling RR peaked in PP (~9.6) then declined; India's was flatter and consistently higher.",
        "India's 5-over rolling RR stayed above 8.5 for most of the chase — remarkable consistency.",
        "WI's rolling RR dips sharply post over 40: 4 wickets derailed the innings.",
        "India's rolling-rate line shows no crisis trough — Rohit\u2013Gill absorbed every challenging spell.",
    ],
    'chase': [
        "India ahead of required rate from ~over 5 onwards — the green zone dominates the chart.",
        "Required rate started at 8.12 RPO; India's actual rate consistently ran at 9.0\u20139.5 RPO.",
        "By over 30, required rate had fallen below 6.0 RPO — the match was effectively decided.",
        f"India completed the chase in over {int(legal2//6)+1}, winning with {300-int(legal2)} balls ({(300-int(legal2))//6}.{(300-int(legal2))%6} overs) to spare.",
    ],
    'contrib': [
        "WI top 3 contributed ~79% of team total — brilliant but dangerously top-heavy.",
        "India top 2 (Rohit + Gill) contributed ~80%; Gill's ~55% slice alone is historic for a chase.",
        "WI middle/lower order added only ~84 runs across 4 partnerships — failed to capitalise.",
        "Both charts reinforce: whoever wins the opening-order battle wins the modern ODI.",
    ],
    'boundary': [
        "Middle overs most boundaries for both: WI 22\u00d74 + 6\u00d76; IND 24\u00d74 + 9\u00d76.",
        "India hit more sixes in every phase (15 total vs WI's 12) — more attacking stroke-play.",
        "WI death boundaries collapsed (7 fours + 1 six in 10 overs) as wickets fell.",
        "India's PP boundary rate (10\u00d74 + 5\u00d76 in 10 overs, 0 wickets) set the foundation perfectly.",
    ],
    'dismissal': [
        "India lost only 2 wickets in 43.3 overs — the dismissal chart is nearly blank.",
        "WI's 7 dismissals spread across multiple types shows India's varied bowling attack.",
        "No single WI batsman anchored the innings after the top 3 departed — middle order brittle.",
        "Dismissal chart visual summary: India barely needed their middle/lower order to feature.",
    ],
}

doc = SimpleDocTemplate(
    OUT_PDF, pagesize=A4,
    leftMargin=1.8*cm, rightMargin=1.8*cm,
    topMargin=1.5*cm,  bottomMargin=1.5*cm,
    title='WI vs IND 2nd ODI 2026 Analysis',
    author='Cricket Analytics'
)

story = [PageBreak()]   # page 1 = cover (pure canvas)

story += [section_hdr('MATCH SNAPSHOT', colors.HexColor('#1A237E')), Spacer(1, 0.3*cm),
          make_match_snapshot(), PageBreak()]

story += [section_hdr('SCORING PROGRESSION', WI_MAROON), Spacer(1,0.2*cm),
          chart_scoring(), inf_box(*INF['scoring']), Spacer(1,0.3*cm),
          chart_rr(), inf_box(*INF['rr']),
          Spacer(1,0.3*cm), chart_rolling_rr(), inf_box(*INF['rolling']), PageBreak()]

story += [section_hdr('CHASE ANALYSIS', IND_BLUE), Spacer(1,0.2*cm),
          chart_required_rate_tracker(), inf_box(*INF['chase']), Spacer(1,0.4*cm),
          mini_hdr('Over-by-Over Comparison', colors.HexColor('#90CAF9')),
          Spacer(1,0.2*cm), make_over_comparison_table(), PageBreak()]

story += [section_hdr('RUNS PER OVER', WI_MAROON), Spacer(1,0.2*cm),
          chart_per_over(), inf_box(*INF['perover']), Spacer(1,0.4*cm),
          section_hdr('PHASE-WISE COMPARISON', IND_BLUE), Spacer(1,0.2*cm),
          chart_phase(), inf_box(*INF['phase']), Spacer(1,0.3*cm),
          mini_hdr('Dot Ball % by Phase', colors.HexColor('#90CAF9')),
          chart_dot_comparison(), inf_box(*INF['dot']), PageBreak()]

story += [section_hdr('PHASE-WISE SCORECARD', WI_MAROON), Spacer(1,0.3*cm),
          mini_hdr(f'{team1}  \u2014  {total1}/{int(wkts1)} in {ov1_str} overs  (Batting First)',
                   colors.HexColor('#FFD700')),
          phase_table(ps1, WI_MAROON), Spacer(1,0.4*cm),
          mini_hdr(f'{team2}  \u2014  {total2}/{int(wkts2)} in {ov2_str} overs  (Batting Second)',
                   IND_TEXT),
          phase_table(ps2, IND_BLUE), PageBreak()]

story += [section_hdr('BATTING SCORECARDS', IND_BLUE), Spacer(1,0.2*cm),
          chart_batsmen(), inf_box(*INF['bat']), Spacer(1,0.3*cm),
          mini_hdr(f'{team1} Batting', colors.HexColor('#FFD700')), bat_table(bs1, WI_MAROON),
          Spacer(1,0.4*cm),
          mini_hdr(f'{team2} Batting', IND_TEXT),   bat_table(bs2, IND_BLUE),
          Spacer(1,0.5*cm),
          mini_hdr('Batsman Contribution  (% of Team Total)', colors.HexColor('#90CAF9')),
          chart_batsman_contribution(), inf_box(*INF['contrib']),
          mini_hdr('Boundary Distribution by Phase', colors.HexColor('#90CAF9')),
          chart_boundary_by_phase(), inf_box(*INF['boundary']), PageBreak()]

story += [section_hdr('FALL OF WICKETS & PARTNERSHIPS', colors.HexColor('#4A148C')),
          Spacer(1,0.2*cm), chart_fow_partnerships(), inf_box(*INF['fow']),
          Spacer(1,0.4*cm),
          mini_hdr(f'{team1} Partnerships', colors.HexColor('#FFD700')),
          make_partnership_table(par1, WI_MAROON),
          Spacer(1,0.3*cm),
          mini_hdr(f'{team2} Partnerships', IND_TEXT),
          make_partnership_table(par2, IND_BLUE), PageBreak()]

story += [section_hdr('BOWLING SCORECARDS', WI_MAROON), Spacer(1,0.2*cm),
          chart_bowling(), inf_box(*INF['bowl']), Spacer(1,0.3*cm),
          chart_eco_vs_wkt(), inf_box(*INF['eco']), Spacer(1,0.3*cm),
          mini_hdr('Dismissal Types', colors.HexColor('#90CAF9')),
          chart_dismissal_types(), inf_box(*INF['dismissal']), Spacer(1,0.3*cm),
          mini_hdr(f'{team1} Bowling  (vs {team2})', colors.HexColor('#FFD700')),
          bowl_table(bw1, WI_MAROON), Spacer(1,0.4*cm),
          mini_hdr(f'{team2} Bowling  (vs {team1})', IND_TEXT),
          bowl_table(bw2, IND_BLUE), PageBreak()]

story += [section_hdr('MATCH ANALYTICS', colors.HexColor('#1A237E')), Spacer(1, 0.3*cm),
          chart_analytics(), inf_box(*INF['analytics']), Spacer(1, 0.4*cm),
          mini_hdr('Match Context at a Glance', colors.HexColor('#90CAF9')),
          Spacer(1, 0.2*cm), make_match_context(), PageBreak()]

story.append(section_hdr('PLAYERS SHINING BRIGHT', GREEN_GOOD))
story.append(Spacer(1,0.3*cm))
shining = auto_shining()
cw_sh = [2.8*cm, 1.4*cm, 2.4*cm, 2.4*cm, 7.4*cm]
shin_rows = []
for name, team, score, metric, impact in shining:
    team_abbr = 'WI' if team==team1 else 'IND'
    tc = WI_MAROON if team==team1 else IND_BLUE
    shin_rows.append([
        P(name),
        P(team_abbr, PS('ta',fontSize=9,fontName='Helvetica-Bold',
                         textColor=tc,alignment=TA_CENTER,leading=12)),
        PC(score), PC(metric), P(impact),
    ])
if not shin_rows:
    shin_rows.append([PC('\u2013'), PC('\u2013'), PC('\u2013'), PC('\u2013'), P('No standout performers')])
shin_tbl = make_table(['Player','Team','Score','Key Metric','Impact'],
                      shin_rows, cw_sh, hdr_bg=GREEN_GOOD)
story += [shin_tbl, PageBreak()]

story.append(section_hdr('KEY UNDERPERFORMERS', RED_BAD))
story.append(Spacer(1,0.3*cm))
pressure = auto_pressure()
cw_pr = [2.8*cm, 1.4*cm, 2.4*cm, 2.4*cm, 7.4*cm]
pres_rows = []
for name, team, score, stat, reason in pressure:
    team_abbr = 'WI' if team==team1 else 'IND'
    tc = WI_MAROON if team==team1 else IND_BLUE
    pres_rows.append([
        P(name),
        P(team_abbr, PS('ta2',fontSize=9,fontName='Helvetica-Bold',
                          textColor=tc,alignment=TA_CENTER,leading=12)),
        PC(score), PC(stat), P(reason),
    ])
if not pres_rows:
    pres_rows.append([PC('\u2013'), PC('\u2013'), PC('\u2013'), PC('\u2013'), P('No underperformers flagged')])
pres_tbl = make_table(['Player','Team','Score/Figures','Stat','Why They Struggled'],
                      pres_rows, cw_pr, hdr_bg=RED_BAD)
story += [pres_tbl, PageBreak()]

story.append(section_hdr('TEAM PHASE ANALYSIS & KEY MISTAKES', WI_MAROON))
story.append(Spacer(1,0.3*cm))
story.append(mini_hdr(f'{team1} (WI) \u2014 What Went Wrong', colors.HexColor('#FFD700')))
cw_m = [2.6*cm, 4.4*cm, 4.2*cm, 5.2*cm]
wi_rows = [
    [PC('Middle\n(Batting)'),
     P('WI were set back by regular wickets in middle overs'),
     P('Wickets fell at key junctures; run rate dipped'),
     P('India kept pressure, WI could not sustain momentum.')],
    [PC('Death\n(Bowling)'),
     P('India exploited death overs freely'),
     P(f'India scored at 9+ RPO in overs 41-50'),
     P('WI lacked a death specialist to restrict boundaries.')],
    [PC('Powerplay\n(Bowling)'),
     P('Could not break opening stand early'),
     P(f'India powerplay: 92/0 \u2014 solid platform'),
     P('Early wickets would have changed chase dynamics.')],
]
wi_mis = make_table(['Phase','Issue','Numbers','Impact'],
                    wi_rows, cw_m, hdr_bg=WI_MAROON, stripe=False)
wi_mis.setStyle(TableStyle([
    ('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.HexColor('#FFF0F5'),WHITE])]))
story += [wi_mis, Spacer(1,0.5*cm)]

IND_TEXT2 = colors.HexColor('#4488FF')
story.append(mini_hdr(f'{team2} (IND) \u2014 Minor Slippages', IND_TEXT2))
ind_rows = [
    [PC('Bowling\n(Economy)'),
     P('WI middle order scored fluently'),
     P(f'WI scored {total1} runs \u2014 highest total in this series'),
     P('India needed early wickets to keep WI below 380.')],
    [PC('Extras'),
     P('Wides and no-balls gifted free runs'),
     P(f'WI extras: {int(inn1["extras"].sum())} total'),
     P('Gifted WI runs without batting effort.')],
]
ind_mis = make_table(['Phase','Issue','Numbers','Impact'],
                     ind_rows, cw_m, hdr_bg=IND_BLUE, stripe=False)
ind_mis.setStyle(TableStyle([
    ('TEXTCOLOR',(0,0),(-1,0), WHITE),
    ('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.HexColor('#EEF4FF'),WHITE])]))
story += [ind_mis, PageBreak()]

story.append(section_hdr('KEY TURNING POINTS', ORANGE_MID))
story.append(Spacer(1,0.3*cm))
cw_tp = [2.8*cm, 5.2*cm, 2.8*cm, 5.6*cm]
tp_rows = auto_turning_points()
tp_tbl = make_table(['Over','Event','Score','Impact'], tp_rows, cw_tp, hdr_bg=ORANGE_MID)
story += [tp_tbl, Spacer(1,0.5*cm)]

story.append(section_hdr('MATCH VERDICT', IND_BLUE))
story.append(Spacer(1,0.3*cm))

top_bat_ind = bs2[0] if bs2 else {'Batsman': 'N/A', 'R': 0, 'B': 0}
verdict_txt = (
    f'<b>{team2}</b> produced a stunning run-chase for the ages, overhauling '
    f'{team1}\'s imposing {total1}/{int(wkts1)} target with {int(winner_wkts)} wickets '
    f'in hand and {300-int(legal2)} balls to spare. '
    f'{fmt_name(top_bat_ind["Batsman"])} led the charge with an extraordinary innings '
    f'of {top_bat_ind["R"]}({top_bat_ind["B"]}), anchoring a clinical chase. '
    f'{team1} posted a formidable total powered by three centurions \u2014 a historic achievement '
    f'\u2014 but India\'s batting depth and firepower proved too much on the day. '
    f'The powerplay battle was crucial: India\'s 92/0 gave them '
    f'a solid foundation, and they never looked back. '
    f'<b>Final: India won by {winner_wkts} wickets.</b>'
)
verdict_box = Table([[Paragraph(verdict_txt,
    PS('verd', fontSize=10, fontName='Helvetica', leading=15,
       textColor=BLACK, alignment=TA_LEFT))]],
    colWidths=[16.4*cm])
verdict_box.setStyle(TableStyle([
    ('BACKGROUND',   (0,0),(-1,-1), GRAY_LIGHT),
    ('TOPPADDING',   (0,0),(-1,-1), 10),
    ('BOTTOMPADDING',(0,0),(-1,-1), 10),
    ('LEFTPADDING',  (0,0),(-1,-1), 12),
    ('RIGHTPADDING', (0,0),(-1,-1), 12),
]))
story += [verdict_box, Spacer(1,0.5*cm),
          Paragraph(f'Player of the Match: <b>{pom}</b>',
              PS('pom', fontSize=12, fontName='Helvetica-Bold',
                 textColor=colors.HexColor('#FF9933'), alignment=TA_CENTER))]

doc.build(story, onFirstPage=cover_canvas, onLaterPages=dark_bg_canvas)
print(f'PDF saved: {OUT_PDF}')
